In [1]:

from pathlib import Path
import duckdb
import pandas as pd

# Identify the repository root.
# Assumes Jupyter's working directory is nba_economics/notebooks.
cwd = Path.cwd()
repo_root = cwd if (cwd / "data" / "canonical").is_dir() else cwd.parent
if not (repo_root / "data" / "canonical").is_dir():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/ directory.")

# Locate our frozen Phase A canonical datasets.
canonical_dir = repo_root / "data" / "canonical"

# Establish an in-memory DuckDB database connection.
# We can execute SQL without running a separate database server.
con = duckdb.connect()

print("Canonical directory:", canonical_dir)
print("Directory exists:", canonical_dir.exists())
print("DuckDB version:", duckdb.__version__)


Canonical directory: /Users/jonathantran/Desktop/nba_economics/data/canonical
Directory exists: True
DuckDB version: 2.0.0


In [2]:

# Map SQL view names to canonical CSV filenames.
sources = {
    "seasons": "seasons.csv",                     # C01
    "team_seasons": "team_seasons.csv",           # C04
    "team_apron_seasons": "team_apron_seasons.csv" # C10
}

# Register each canonical CSV as a DuckDB SQL view.
for table_name, filename in sources.items():

    # Construct the complete path to the CSV.
    csv_path = canonical_dir / filename

    # Verify the file exists before creating the view.
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing canonical file: {csv_path}")

    # Convert the path to a SQL string literal.
    # Escape single quotes so paths containing apostrophes remain valid.
    sql_path = str(csv_path).replace("'", "''")

    # Create a view pointing to the existing CSV.
    # DuckDB will read the file when we query the view.
    con.execute(f"""
        CREATE OR REPLACE VIEW {table_name} AS
        SELECT *
        FROM read_csv_auto('{sql_path}')
    """)

    # Fully complete the CREATE VIEW statement before the next query.
    con.fetchall()

    print(f"Registered SQL view: {table_name}")


Registered SQL view: seasons
Registered SQL view: team_seasons
Registered SQL view: team_apron_seasons


In [3]:

# ============================================================
# A01 — Inspect registered canonical tables
# ============================================================

# Loop through each canonical table registered in DuckDB.
for table_name in sources:

    # Run the row-count query and convert the result to a DataFrame.
    # This avoids leaving an unfinished DuckDB relation/result.
    row_count_df = con.sql(
        f"SELECT COUNT(*) AS n_rows FROM {table_name}"
    ).df()

    # Extract the single row-count value.
    row_count = int(row_count_df["n_rows"].iloc[0])

    # Inspect column names and DuckDB data types.
    # The previous query has already been materialized.
    columns = con.sql(
        f"DESCRIBE {table_name}"
    ).df()

    print(f"\n{'=' * 50}")
    print(f"TABLE: {table_name}")
    print(f"ROWS: {row_count}")

    # Display the schema for each canonical table.
    display(columns[["column_name", "column_type"]])



TABLE: seasons
ROWS: 26


,column_name,column_type
0,season,VARCHAR
1,Year,BIGINT
2,Salary Cap,BIGINT
3,Salary Cap Floor Pct,DOUBLE
4,Salary Cap Floor,BIGINT
5,Luxury Tax,DOUBLE
6,1st Apron,DOUBLE
7,2nd Apron,DOUBLE
8,BAE,BIGINT
9,Non-Taxpayer MLE,BIGINT



TABLE: team_seasons
ROWS: 776


,column_name,column_type
0,team_name,VARCHAR
1,wins,BIGINT
2,losses,BIGINT
3,win_pct,DOUBLE
4,games_behind,DOUBLE
5,points_per_game,DOUBLE
6,opponent_points_per_game,DOUBLE
7,simple_rating_system,DOUBLE
8,conference,VARCHAR
9,season,VARCHAR



TABLE: team_apron_seasons
ROWS: 450


,column_name,column_type
0,franchise_id,BIGINT
1,season,VARCHAR
2,total_apron_allocations,BIGINT
3,first_apron_threshold,BIGINT
4,second_apron_threshold,DOUBLE
5,first_apron_space,BIGINT
6,second_apron_space,DOUBLE
7,above_first_apron,BOOLEAN
8,above_second_apron,BOOLEAN
9,first_apron_hard_capped,BOOLEAN


In [4]:

# Define each table's expected primary key.
keys = {
    "seasons": ["season"],
    "team_seasons": ["franchise_id", "season"],
    "team_apron_seasons": ["franchise_id", "season"]
}

for table_name, key_columns in keys.items():

    # Combine key columns into a SQL GROUP BY expression.
    key_sql = ", ".join(key_columns)

    # Define SQL to find duplicate key combinations.
    sql_duplicates = f"""
    SELECT
        {key_sql},
        COUNT(*) AS n
    FROM {table_name}
    GROUP BY {key_sql}
    HAVING COUNT(*) > 1
    """

    # Execute the SQL and return a pandas DataFrame.
    duplicates = con.sql(sql_duplicates).df()

    # Construct a SQL condition to identify missing key values.
    null_condition = " OR ".join(
        f"{column} IS NULL" for column in key_columns
    )

    # Define SQL to count rows with missing keys.
    sql_missing = f"""
    SELECT
        COUNT(*) AS n
    FROM {table_name}
    WHERE {null_condition}
    """

    # Execute SQL and extract the count from the DataFrame.
    missing_keys = con.sql(sql_missing).df()["n"].iloc[0]

    print(f"\nTABLE: {table_name}")
    print(f"Duplicate key combinations: {len(duplicates)}")
    print(f"Rows with missing keys: {missing_keys}")



TABLE: seasons
Duplicate key combinations: 0
Rows with missing keys: 0

TABLE: team_seasons
Duplicate key combinations: 0
Rows with missing keys: 0

TABLE: team_apron_seasons
Duplicate key combinations: 0
Rows with missing keys: 0


In [5]:

# Define the SQL query separately from its execution.
sql_join_coverage = """
SELECT
    COUNT(*) AS total_team_seasons,

    -- Count C04 rows without a matching C01 season.
    COUNT(*) FILTER (
        WHERE s.season IS NULL
    ) AS missing_season_match,

    -- Count C04 rows without a matching C10 apron record.
    COUNT(*) FILTER (
        WHERE a.franchise_id IS NULL
    ) AS missing_apron_match

FROM team_seasons AS t

LEFT JOIN seasons AS s
    ON t.season = s.season

LEFT JOIN team_apron_seasons AS a
    ON t.franchise_id = a.franchise_id
   AND t.season = a.season
"""

# Execute SQL and return the results as a pandas DataFrame.
join_coverage_df = con.sql(sql_join_coverage).df()

display(join_coverage_df)


,total_team_seasons,missing_season_match,missing_apron_match
0,776,0,326


In [6]:

# Define the SQL that constructs our A01 base table.
sql_a01_base = """
CREATE OR REPLACE TEMP VIEW a01_base AS

SELECT
    -- Preserve every C04 franchise-season column.
    t.*,

    -- Add season-level context from C01.
    s."Year" AS season_year,
    s."Salary Cap" AS salary_cap,
    s."Salary Cap Floor Pct" AS salary_cap_floor_pct,
    s."Salary Cap Floor" AS salary_cap_floor,
    s."Luxury Tax" AS league_luxury_tax_threshold,
    s."1st Apron" AS league_first_apron,
    s."2nd Apron" AS league_second_apron,
    s.cba_id,

    -- Add franchise-specific apron information from C10.
    a.total_apron_allocations,
    a.first_apron_threshold,
    a.second_apron_threshold,
    a.first_apron_space,
    a.second_apron_space,
    a.above_first_apron,
    a.above_second_apron,
    a.first_apron_hard_capped,
    a.second_apron_hard_capped,
    a.second_apron_applicable,
    a.verification_status AS apron_verification_status,
    a.source_coverage AS apron_source_coverage

FROM team_seasons AS t

-- Every franchise-season should match one season record.
LEFT JOIN seasons AS s
    ON t.season = s.season

-- C10 covers only part of our historical period.
LEFT JOIN team_apron_seasons AS a
    ON t.franchise_id = a.franchise_id
   AND t.season = a.season
"""

# Execute the SQL statement to create the DuckDB view.
# This does not create a pandas DataFrame yet.
con.execute(sql_a01_base).fetchall()

print("Created DuckDB view: a01_base")


Created DuckDB view: a01_base


In [7]:

# Check the resulting table's row count and unique keys.
sql_a01_validation = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (franchise_id, season)) AS unique_franchise_seasons,

    -- Count observations with a matched season-level salary cap.
    COUNT(salary_cap) AS rows_with_salary_cap,

    -- Count observations with C10 apron allocations.
    COUNT(total_apron_allocations) AS rows_with_apron_data

FROM a01_base
"""

# Convert the SQL result into a pandas DataFrame.
a01_validation_df = con.sql(sql_a01_validation).df()

display(a01_validation_df)

# Use Python assertions to enforce our expected structure.
assert a01_validation_df["total_rows"].iloc[0] == 776
assert a01_validation_df["unique_franchise_seasons"].iloc[0] == 776
assert a01_validation_df["rows_with_salary_cap"].iloc[0] == 776
assert a01_validation_df["rows_with_apron_data"].iloc[0] == 450

print("A01 v0 structural validation passed.")


,total_rows,unique_franchise_seasons,rows_with_salary_cap,rows_with_apron_data
0,776,776,776,450


A01 v0 structural validation passed.


In [8]:

# Define the SQL for A01 v1A.
# We use a TEMP VIEW so our canonical CSVs remain unchanged.

sql_a01_v1a = """
CREATE OR REPLACE TEMP VIEW a01_v1a AS

SELECT
    -- Retain all existing columns from A01 base.
    b.*,

    -- 1. Team payroll relative to the league salary cap.
    b.team_payroll * 1.0
        / NULLIF(b.salary_cap, 0)
        AS payroll_to_cap_ratio,

    -- 2. Reported cap hit relative to the salary cap.
    b.salary_cap_hit * 1.0
        / NULLIF(b.salary_cap, 0)
        AS cap_hit_to_cap_ratio,

    -- 3. Taxable payroll relative to the luxury-tax threshold.
    b.taxable_payroll * 1.0
        / NULLIF(b.league_luxury_tax_threshold, 0)
        AS taxable_payroll_to_tax_ratio,

    -- 4. Apron allocations relative to the first-apron threshold.
    b.total_apron_allocations * 1.0
        / NULLIF(b.first_apron_threshold, 0)
        AS first_apron_allocation_ratio,

    -- 5. Second-apron ratio, only when applicable.
    CASE
        WHEN b.second_apron_applicable = TRUE
        THEN b.total_apron_allocations * 1.0
            / NULLIF(b.second_apron_threshold, 0)
        ELSE NULL
    END AS second_apron_allocation_ratio

FROM a01_base AS b
"""

# Execute the CREATE VIEW statement.
con.execute(sql_a01_v1a).fetchall()

print("A01 v1A view created successfully.")


A01 v1A view created successfully.


In [9]:

# Define a validation query for our five engineered ratios.

sql_v1a_validation = """
SELECT
    -- Structural integrity: A01 must retain its original grain.
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

    -- Feature coverage: COUNT(column) excludes NULL values.
    COUNT(payroll_to_cap_ratio) AS n_payroll_cap,
    COUNT(cap_hit_to_cap_ratio) AS n_cap_hit_cap,
    COUNT(taxable_payroll_to_tax_ratio) AS n_taxable_tax,
    COUNT(first_apron_allocation_ratio) AS n_first_apron,
    COUNT(second_apron_allocation_ratio) AS n_second_apron,

    -- Basic numerical checks.
    ROUND(MIN(payroll_to_cap_ratio), 3) AS min_payroll_cap,
    ROUND(MAX(payroll_to_cap_ratio), 3) AS max_payroll_cap,

    ROUND(MIN(taxable_payroll_to_tax_ratio), 3) AS min_taxable_tax,
    ROUND(MAX(taxable_payroll_to_tax_ratio), 3) AS max_taxable_tax

FROM a01_v1a
"""

# Execute SQL and convert the results into a pandas DataFrame.
v1a_audit = con.sql(sql_v1a_validation).df()

display(v1a_audit)

# Confirm that no franchise-season rows were lost or duplicated.
assert v1a_audit["total_rows"].iloc[0] == 776
assert v1a_audit["unique_keys"].iloc[0] == 776

print("A01 v1A structural validation passed.")


,total_rows,unique_keys,n_payroll_cap,n_cap_hit_cap,n_taxable_tax,n_first_apron,n_second_apron,min_payroll_cap,max_payroll_cap,min_taxable_tax,max_taxable_tax
0,776,776,776,776,450,450,90,0.545,2.622,0.521,1.435


A01 v1A structural validation passed.


In [10]:

# Inspect the ten highest-payroll teams for the 2023-24 season.

sql_v1a_sample = """
SELECT
    season,
    team_name,

    ROUND(payroll_to_cap_ratio, 3) AS payroll_cap,
    ROUND(cap_hit_to_cap_ratio, 3) AS cap_hit_cap,
    ROUND(taxable_payroll_to_tax_ratio, 3) AS taxable_tax,
    ROUND(first_apron_allocation_ratio, 3) AS first_apron,
    ROUND(second_apron_allocation_ratio, 3) AS second_apron

FROM a01_v1a

WHERE season = '2023-2024'

ORDER BY team_payroll DESC

LIMIT 10
"""

v1a_sample = con.sql(sql_v1a_sample).df()

display(v1a_sample)


,season,team_name,payroll_cap,cap_hit_cap,taxable_tax,first_apron,second_apron
0,2023-2024,Golden State Warriors,1.539,1.539,1.244,1.206,1.137
1,2023-2024,Los Angeles Clippers,1.480,1.480,1.210,1.161,1.094
2,2023-2024,Phoenix Suns,1.425,1.425,1.158,1.115,1.051
3,2023-2024,Milwaukee Bucks,1.377,1.377,1.102,1.071,1.010
4,2023-2024,Boston Celtics,1.374,1.374,1.119,1.094,1.032
5,2023-2024,Denver Nuggets,1.330,1.330,1.070,1.033,0.974
6,2023-2024,Miami Heat,1.302,1.302,1.057,1.028,0.969
7,2023-2024,Los Angeles Lakers,1.249,1.249,1.017,0.980,0.924
8,2023-2024,Dallas Mavericks,1.233,1.233,0.996,0.964,0.909
9,2023-2024,New Orleans Pelicans,1.231,1.231,0.998,0.957,0.902


In [11]:

# Inspect the financial amounts, thresholds, and normalized ratios
# for the five highest-payroll teams in 2023-2024.
# This changes only our inspection output, NOT the underlying A01 view.

sql_v1a_interpretation = """
SELECT
    season,
    team_name,

    -- Payroll compared with the league salary cap.
    ROUND(team_payroll / 1000000.0, 2) AS payroll_m,
    ROUND(salary_cap / 1000000.0, 2) AS salary_cap_m,
    ROUND(payroll_to_cap_ratio * 100, 1) AS payroll_cap_pct,

    -- Reported cap hit compared with the league salary cap.
    ROUND(salary_cap_hit / 1000000.0, 2) AS cap_hit_m,
    ROUND(cap_hit_to_cap_ratio * 100, 1) AS cap_hit_cap_pct,

    -- Taxable payroll compared with the luxury-tax threshold.
    ROUND(taxable_payroll / 1000000.0, 2) AS taxable_payroll_m,
    ROUND(league_luxury_tax_threshold / 1000000.0, 2)
        AS tax_threshold_m,
    ROUND(taxable_payroll_to_tax_ratio * 100, 1)
        AS taxable_tax_pct,

    -- Apron allocations compared with first- and second-apron thresholds.
    ROUND(total_apron_allocations / 1000000.0, 2)
        AS apron_allocations_m,
    ROUND(first_apron_threshold / 1000000.0, 2)
        AS first_apron_m,
    ROUND(first_apron_allocation_ratio * 100, 1)
        AS first_apron_pct,

    ROUND(second_apron_threshold / 1000000.0, 2)
        AS second_apron_m,
    ROUND(second_apron_allocation_ratio * 100, 1)
        AS second_apron_pct

FROM a01_v1a

WHERE season = '2023-2024'

ORDER BY team_payroll DESC

LIMIT 5
"""

display(con.sql(sql_v1a_interpretation).df())


,season,team_name,payroll_m,salary_cap_m,payroll_cap_pct,cap_hit_m,cap_hit_cap_pct,taxable_payroll_m,tax_threshold_m,taxable_tax_pct,apron_allocations_m,first_apron_m,first_apron_pct,second_apron_m,second_apron_pct
0,2023-2024,Golden State Warriors,209.35,136.02,153.9,209.35,153.9,205.57,165.29,124.4,207.81,172.35,120.6,182.79,113.7
1,2023-2024,Los Angeles Clippers,201.37,136.02,148.0,201.37,148.0,200.06,165.29,121.0,200.06,172.35,116.1,182.79,109.4
2,2023-2024,Phoenix Suns,193.84,136.02,142.5,193.84,142.5,191.33,165.29,115.8,192.18,172.35,111.5,182.79,105.1
3,2023-2024,Milwaukee Bucks,187.35,136.02,137.7,187.35,137.7,182.23,165.29,110.2,184.57,172.35,107.1,182.79,101.0
4,2023-2024,Boston Celtics,186.94,136.02,137.4,186.94,137.4,184.94,165.29,111.9,188.58,172.35,109.4,182.79,103.2


In [12]:

# Compare reported payroll, actual tax, estimated tax,
# and total estimated cost for a recent season.
# This is a diagnostic SELECT, not a new engineered feature.

sql_cost_definition_check = """
SELECT
    season,
    team_name,

    ROUND(team_payroll / 1000000.0, 2) AS payroll_m,
    ROUND(tax_paid / 1000000.0, 2) AS tax_paid_m,
    ROUND(estimated_tax_bill / 1000000.0, 2) AS estimated_tax_m,
    ROUND(total_estimated_cost / 1000000.0, 2) AS total_estimated_cost_m,

    -- Test whether total estimated cost matches
    -- payroll plus the estimated tax bill.
    ROUND(
        (total_estimated_cost - team_payroll - estimated_tax_bill)
        / 1000000.0, 2
    ) AS estimated_cost_difference_m

FROM a01_v1a

WHERE total_estimated_cost IS NOT NULL

ORDER BY season DESC, team_payroll DESC

LIMIT 15
"""

display(con.sql(sql_cost_definition_check).df())


,season,team_name,payroll_m,tax_paid_m,estimated_tax_m,total_estimated_cost_m,estimated_cost_difference_m
0,2025-2026,Cleveland Cavaliers,211.86,NaN,69.44,281.93,0.63
1,2025-2026,New York Knicks,207.05,NaN,44.44,251.86,0.38
2,2025-2026,Golden State Warriors,205.12,NaN,67.91,273.02,-0.01
3,2025-2026,Houston Rockets,199.78,NaN,7.17,201.95,-5.01
4,2025-2026,Los Angeles Lakers,197.15,NaN,22.27,217.45,-1.96
5,2025-2026,Minnesota Timberwolves,196.04,NaN,6.14,200.08,-2.09
6,2025-2026,Philadelphia 76ers,194.67,NaN,NaN,187.88,NaN
7,2025-2026,Sacramento Kings,193.33,NaN,NaN,187.67,NaN
8,2025-2026,Los Angeles Clippers,191.94,NaN,6.67,196.78,-1.82
9,2025-2026,Toronto Raptors,191.61,NaN,NaN,187.21,NaN


In [13]:

# V1B diagnostic: test plausible arithmetic definitions
# of C04's total_estimated_cost.
#
# This does not change the analytical dataset.
# It only compares existing source columns.

sql_total_cost_audit = """
SELECT
    COUNT(*) AS total_rows,

    -- Coverage of each relevant source field.
    COUNT(total_estimated_cost) AS n_total_cost,
    COUNT(team_payroll) AS n_team_payroll,
    COUNT(taxable_payroll) AS n_taxable_payroll,
    COUNT(tax_paid) AS n_tax_paid,
    COUNT(estimated_tax_bill) AS n_estimated_tax,

    -- Count rows where total cost equals payroll + estimated tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND estimated_tax_bill IS NOT NULL
          AND ABS(
              total_estimated_cost
              - team_payroll
              - estimated_tax_bill
          ) < 1
    ) AS matches_payroll_plus_est_tax,

    -- Count rows where total cost equals payroll + reported tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND tax_paid IS NOT NULL
          AND ABS(
              total_estimated_cost
              - team_payroll
              - tax_paid
          ) < 1
    ) AS matches_payroll_plus_tax_paid,

    -- Count rows where total cost equals taxable payroll + estimated tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND estimated_tax_bill IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - estimated_tax_bill
          ) < 1
    ) AS matches_taxable_plus_est_tax,

    -- Count rows where total cost equals taxable payroll + reported tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND tax_paid IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - tax_paid
          ) < 1
    ) AS matches_taxable_plus_tax_paid

FROM a01_v1a
"""

display(con.sql(sql_total_cost_audit).df())


,total_rows,n_total_cost,n_team_payroll,n_taxable_payroll,n_tax_paid,n_estimated_tax,matches_payroll_plus_est_tax,matches_payroll_plus_tax_paid,matches_taxable_plus_est_tax,matches_taxable_plus_tax_paid
0,776,450,776,450,268,94,0,0,94,0


In [14]:

# Inspect reported total cost alongside the candidate components.
# COALESCE is deliberately avoided here so missing tax
# values are not silently treated as zero.

sql_total_cost_examples = """
SELECT
    season,
    team_name,

    ROUND(team_payroll / 1000000.0, 2) AS payroll_m,
    ROUND(taxable_payroll / 1000000.0, 2) AS taxable_m,
    ROUND(tax_paid / 1000000.0, 2) AS tax_paid_m,
    ROUND(estimated_tax_bill / 1000000.0, 2) AS estimated_tax_m,
    ROUND(total_estimated_cost / 1000000.0, 2) AS total_cost_m,

    ROUND(
        (total_estimated_cost - team_payroll)
        / 1000000.0, 2
    ) AS cost_minus_payroll_m,

    ROUND(
        (total_estimated_cost - taxable_payroll)
        / 1000000.0, 2
    ) AS cost_minus_taxable_m

FROM a01_v1a

WHERE total_estimated_cost IS NOT NULL

ORDER BY season DESC, team_payroll DESC

LIMIT 15
"""

display(con.sql(sql_total_cost_examples).df())


,season,team_name,payroll_m,taxable_m,tax_paid_m,estimated_tax_m,total_cost_m,cost_minus_payroll_m,cost_minus_taxable_m
0,2025-2026,Cleveland Cavaliers,211.86,212.49,NaN,69.44,281.93,70.07,69.44
1,2025-2026,New York Knicks,207.05,207.42,NaN,44.44,251.86,44.81,44.44
2,2025-2026,Golden State Warriors,205.12,205.11,NaN,67.91,273.02,67.90,67.91
3,2025-2026,Houston Rockets,199.78,194.77,NaN,7.17,201.95,2.16,7.17
4,2025-2026,Los Angeles Lakers,197.15,195.18,NaN,22.27,217.45,20.30,22.27
5,2025-2026,Minnesota Timberwolves,196.04,193.94,NaN,6.14,200.08,4.04,6.14
6,2025-2026,Philadelphia 76ers,194.67,187.88,NaN,NaN,187.88,-6.78,0.00
7,2025-2026,Sacramento Kings,193.33,187.67,NaN,NaN,187.67,-5.66,0.00
8,2025-2026,Los Angeles Clippers,191.94,190.12,NaN,6.67,196.78,4.84,6.67
9,2025-2026,Toronto Raptors,191.61,187.21,NaN,NaN,187.21,-4.40,0.00


In [15]:

# Check whether total estimated cost consistently equals:
# taxable payroll + estimated tax (treating missing estimated tax as zero).
#
# COALESCE(x, 0) substitutes zero when x is NULL.
# We use it here to TEST the source's apparent construction rule,
# not to assert that every missing tax estimate represents a true zero liability.

sql_cost_final_check = """
SELECT
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
    ) AS n_total_cost,

    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND estimated_tax_bill IS NULL
    ) AS n_missing_estimated_tax,

    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - COALESCE(estimated_tax_bill, 0)
          ) < 1
    ) AS n_formula_matches,

    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - COALESCE(estimated_tax_bill, 0)
          ) >= 1
    ) AS n_formula_mismatches

FROM a01_v1a
"""

display(con.sql(sql_cost_final_check).df())


,n_total_cost,n_missing_estimated_tax,n_formula_matches,n_formula_mismatches
0,450,356,450,0


In [16]:

# Step 12: Build A01 v1B using the existing v1A view.
# Preserve all v1A columns and add six financial-exposure features.

sql_a01_v1b = """
CREATE OR REPLACE TEMP VIEW a01_v1b AS

SELECT
    b.*,

    -- 1. Signed distance from the luxury-tax threshold.
    -- Positive = above; negative = below.
    b.taxable_payroll
        - b.league_luxury_tax_threshold
        AS tax_threshold_distance,

    -- 2. Signed distance from the first apron.
    b.total_apron_allocations
        - b.first_apron_threshold
        AS first_apron_distance,

    -- 3. Signed distance from the second apron.
    -- Only calculate where the second apron applies.
    CASE
        WHEN b.second_apron_applicable = TRUE
        THEN b.total_apron_allocations
            - b.second_apron_threshold
        ELSE NULL
    END AS second_apron_distance,

    -- 4. Above-tax indicator.
    -- Missing inputs produce NULL, not FALSE.
    CASE
        WHEN b.taxable_payroll IS NULL
          OR b.league_luxury_tax_threshold IS NULL
        THEN NULL
        ELSE b.taxable_payroll > b.league_luxury_tax_threshold
    END AS above_tax_threshold,

    -- 5. Reported tax paid relative to team payroll.
    -- Do not replace missing tax payments with zero.
    b.tax_paid * 1.0
        / NULLIF(b.team_payroll, 0)
        AS tax_paid_to_payroll_ratio,

    -- 6. Estimated total cost relative to taxable payroll.
    -- Both source fields belong to the same accounting framework.
    b.total_estimated_cost * 1.0
        / NULLIF(b.taxable_payroll, 0)
        AS estimated_cost_to_taxable_ratio

FROM a01_v1a AS b
"""

con.execute(sql_a01_v1b).fetchall()

print("A01 v1B view created successfully.")


A01 v1B view created successfully.


In [17]:

# Validate the franchise-season grain and the six new features.

sql_v1b_validation = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

    -- Coverage for each newly engineered feature.
    COUNT(tax_threshold_distance) AS n_tax_distance,
    COUNT(first_apron_distance) AS n_first_distance,
    COUNT(second_apron_distance) AS n_second_distance,
    COUNT(above_tax_threshold) AS n_above_tax,
    COUNT(tax_paid_to_payroll_ratio) AS n_tax_paid_ratio,
    COUNT(estimated_cost_to_taxable_ratio) AS n_cost_ratio,

    -- Verify the Boolean agrees with the signed distance.
    COUNT(*) FILTER (
        WHERE above_tax_threshold IS NOT NULL
          AND above_tax_threshold
              IS DISTINCT FROM (tax_threshold_distance > 0)
    ) AS tax_flag_mismatches,

    -- Check for unexpected negative tax payments.
    COUNT(*) FILTER (
        WHERE tax_paid_to_payroll_ratio < 0
    ) AS negative_tax_ratios,

    -- Inspect the cost ratio's numerical range.
    ROUND(MIN(estimated_cost_to_taxable_ratio), 3)
        AS min_cost_ratio,
    ROUND(MAX(estimated_cost_to_taxable_ratio), 3)
        AS max_cost_ratio

FROM a01_v1b
"""

v1b_audit = con.sql(sql_v1b_validation).df()
display(v1b_audit)

# Structural checks.
assert v1b_audit["total_rows"].iloc[0] == 776
assert v1b_audit["unique_keys"].iloc[0] == 776

# Consistency checks.
assert v1b_audit["tax_flag_mismatches"].iloc[0] == 0
assert v1b_audit["negative_tax_ratios"].iloc[0] == 0

print("A01 v1B structural and consistency checks passed.")


,total_rows,unique_keys,n_tax_distance,n_first_distance,n_second_distance,n_above_tax,n_tax_paid_ratio,n_cost_ratio,tax_flag_mismatches,negative_tax_ratios,min_cost_ratio,max_cost_ratio
0,776,776,450,450,90,450,268,450,0,0,1.0,1.969


A01 v1B structural and consistency checks passed.


In [18]:

# Inspect financial threshold exposure for the highest-taxable-payroll
# teams in 2023-2024. Dollar values are displayed in millions.
#
# Percentage distances are calculated for presentation only;
# they are not stored as additional A01 features.

sql_v1b_inspection = """
SELECT
    season,
    team_name,

    -- Luxury-tax position.
    ROUND(taxable_payroll / 1e6, 2) AS taxable_m,
    ROUND(league_luxury_tax_threshold / 1e6, 2) AS tax_line_m,
    ROUND(tax_threshold_distance / 1e6, 2) AS tax_distance_m,
    ROUND((taxable_payroll_to_tax_ratio - 1) * 100, 1)
        AS tax_distance_pct,
    above_tax_threshold,

    -- First-apron position.
    ROUND(total_apron_allocations / 1e6, 2) AS allocations_m,
    ROUND(first_apron_threshold / 1e6, 2) AS first_apron_m,
    ROUND(first_apron_distance / 1e6, 2) AS first_distance_m,
    ROUND((first_apron_allocation_ratio - 1) * 100, 1)
        AS first_distance_pct,

    -- Second-apron position.
    ROUND(second_apron_threshold / 1e6, 2) AS second_apron_m,
    ROUND(second_apron_distance / 1e6, 2) AS second_distance_m,
    ROUND((second_apron_allocation_ratio - 1) * 100, 1)
        AS second_distance_pct,

    -- Reported tax and estimated total-cost burdens.
    ROUND(tax_paid / 1e6, 2) AS tax_paid_m,
    ROUND(tax_paid_to_payroll_ratio * 100, 1) AS tax_paid_pct,
    ROUND(total_estimated_cost / 1e6, 2) AS total_cost_m,
    ROUND((estimated_cost_to_taxable_ratio - 1) * 100, 1)
        AS estimated_cost_uplift_pct

FROM a01_v1b

WHERE season = '2023-2024'

ORDER BY taxable_payroll DESC

LIMIT 10
"""

display(con.sql(sql_v1b_inspection).df())


,season,team_name,taxable_m,tax_line_m,tax_distance_m,tax_distance_pct,above_tax_threshold,allocations_m,first_apron_m,first_distance_m,first_distance_pct,second_apron_m,second_distance_m,second_distance_pct,tax_paid_m,tax_paid_pct,total_cost_m,estimated_cost_uplift_pct
0,2023-2024,Golden State Warriors,205.57,165.29,40.28,24.4,True,207.81,172.35,35.47,20.6,182.79,25.02,13.7,NaN,NaN,382.45,86.0
1,2023-2024,Los Angeles Clippers,200.06,165.29,34.77,21.0,True,200.06,172.35,27.71,16.1,182.79,17.27,9.4,NaN,NaN,342.47,71.2
2,2023-2024,Phoenix Suns,191.33,165.29,26.04,15.8,True,192.18,172.35,19.84,11.5,182.79,9.39,5.1,NaN,NaN,259.49,35.6
3,2023-2024,Boston Celtics,184.94,165.29,19.64,11.9,True,188.58,172.35,16.23,9.4,182.79,5.79,3.2,NaN,NaN,228.78,23.7
4,2023-2024,Milwaukee Bucks,182.23,165.29,16.93,10.2,True,184.57,172.35,12.23,7.1,182.79,1.78,1.0,NaN,NaN,234.19,28.5
5,2023-2024,Denver Nuggets,176.89,165.29,11.60,7.0,True,178.09,172.35,5.74,3.3,182.79,-4.70,-2.6,NaN,NaN,197.13,11.4
6,2023-2024,Miami Heat,174.72,165.29,9.43,5.7,True,177.20,172.35,4.85,2.8,182.79,-5.60,-3.1,NaN,NaN,189.98,8.7
7,2023-2024,Los Angeles Lakers,168.06,165.29,2.76,1.7,True,168.96,172.35,-3.39,-2.0,182.79,-13.84,-7.6,NaN,NaN,174.96,4.1
8,2023-2024,New Orleans Pelicans,164.98,165.29,-0.31,-0.2,False,164.90,172.35,-7.44,-4.3,182.79,-17.89,-9.8,NaN,NaN,164.98,0.0
9,2023-2024,Dallas Mavericks,164.63,165.29,-0.66,-0.4,False,166.13,172.35,-6.21,-3.6,182.79,-16.66,-9.1,NaN,NaN,164.63,0.0


A01 v1B — Financial Threshold Exposure

Financial threshold exposure: Taxable payroll and apron allocations are source-reported accounting measurements. Signed threshold distances indicate reported financial positioning, not necessarily official end-of-season assessments or legal hard-cap status.

Tax coverage: tax_paid contains 268 non-null observations, while estimated_tax_bill contains 94. Missing tax values are preserved as missing rather than automatically interpreted as zero.

Estimated total cost: Across 450 observations, total_estimated_cost equals taxable payroll plus the reported estimated-tax component, with missing estimates treated as zero in the source-derived construction. This does not establish that missing estimates represent zero actual liability.

Apron restrictions: Existing C10 hard-cap indicators are preserved with their verification limitations. Apron positioning does not independently establish whether a franchise was hard-capped.

In [19]:

# Inspect the competitive-outcome columns already available in C04.
# This is read-only: it does not change C04 or any A01 view.

sql_c04_competitive_columns = """
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'team_seasons'
  AND (
      column_name ILIKE '%win%'
      OR column_name ILIKE '%loss%'
      OR column_name ILIKE '%game%'
      OR column_name ILIKE '%playoff%'
      OR column_name ILIKE '%round%'
      OR column_name ILIKE '%final%'
      OR column_name ILIKE '%champ%'
      OR column_name ILIKE '%seed%'
  )
ORDER BY ordinal_position
"""

display(con.sql(sql_c04_competitive_columns).df())


,column_name,data_type
0,wins,BIGINT
1,losses,BIGINT
2,win_pct,DOUBLE
3,games_behind,DOUBLE
4,points_per_game,DOUBLE
5,opponent_points_per_game,DOUBLE
6,pythagorean_wins,BIGINT
7,pythagorean_losses,BIGINT
8,attendance_per_game,DOUBLE
9,seed,DOUBLE


In [20]:

# Inspect existing playoff-outcome values and coverage.
# This helps us define milestone Booleans without guessing
# how C04 represents playoff advancement.

sql_v1c_playoff_inspection = """
SELECT
    furthest_round,
    round_score,

    COUNT(*) AS n_team_seasons,

    COUNT(playoff_series_won) AS n_series_values,

    MIN(playoff_series_won) AS min_series_won,
    MAX(playoff_series_won) AS max_series_won,

    COUNT(*) FILTER (
        WHERE made_playoffs = TRUE
    ) AS n_playoff_teams,

    COUNT(*) FILTER (
        WHERE made_playoffs = FALSE
    ) AS n_non_playoff_teams,

    COUNT(*) FILTER (
        WHERE made_playoffs IS NULL
    ) AS n_missing_playoff_flags

FROM a01_v1b

GROUP BY
    furthest_round,
    round_score

ORDER BY
    round_score,
    furthest_round
"""

display(con.sql(sql_v1c_playoff_inspection).df())


,furthest_round,round_score,n_team_seasons,n_series_values,min_series_won,max_series_won,n_playoff_teams,n_non_playoff_teams,n_missing_playoff_flags
0,First Round,1.0,208,208,0,0,208,0,0
1,Conference Semifinals,2.0,104,104,1,1,104,0,0
2,Conference Finals,3.0,52,52,2,2,52,0,0
3,NBA Finals,4.0,52,52,3,4,52,0,0
4,NaN,NaN,360,360,0,0,0,360,0


In [21]:

# V1C: confirm regular-season outcome coverage and
# whether Pythagorean wins are usable across the dataset.

sql_v1c_regular_season_audit = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(wins) AS n_wins,
    COUNT(losses) AS n_losses,
    COUNT(win_pct) AS n_win_pct,
    COUNT(pythagorean_wins) AS n_pythagorean_wins,

    -- Check for impossible or inconsistent regular-season values.
    COUNT(*) FILTER (
        WHERE wins < 0 OR losses < 0
    ) AS negative_records,

    COUNT(*) FILTER (
        WHERE wins IS NOT NULL
          AND losses IS NOT NULL
          AND wins + losses = 0
    ) AS zero_game_records,

    -- Compare the canonical winning percentage with
    -- the percentage calculated from wins and losses.
    COUNT(*) FILTER (
        WHERE wins IS NOT NULL
          AND losses IS NOT NULL
          AND win_pct IS NOT NULL
          AND wins + losses > 0
          AND ABS(
              win_pct - wins * 1.0 / (wins + losses)
          ) > 0.001
    ) AS win_pct_mismatches,

    MIN(wins - pythagorean_wins) AS min_wins_above_pyth,
    MAX(wins - pythagorean_wins) AS max_wins_above_pyth

FROM a01_v1b
"""

display(con.sql(sql_v1c_regular_season_audit).df())


,total_rows,n_wins,n_losses,n_win_pct,n_pythagorean_wins,negative_records,zero_game_records,win_pct_mismatches,min_wins_above_pyth,max_wins_above_pyth
0,776,776,776,776,776,0,0,0,-9,8


In [22]:

# A01 v1C — Competitive Outcomes & Spending Efficiency
#
# Build on v1B, preserving all existing canonical columns
# and previously engineered financial features.

sql_a01_v1c = """
CREATE OR REPLACE TEMP VIEW a01_v1c AS
SELECT
    b.*,

    -- 1. Actual regular-season games played.
    -- Useful when comparing seasons with different game counts.
    b.wins + b.losses AS games_played,

    -- 2. Reached at least the conference semifinals (second round).
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.round_score >= 2
    END AS reached_second_round,

    -- 3. Reached at least the conference finals.
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.round_score >= 3
    END AS reached_conference_finals,

    -- 4. Reached the NBA Finals.
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.round_score >= 4
    END AS reached_nba_finals,

    -- 5. Won the NBA championship.
    -- Four playoff series wins identify the champion in our data.
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.playoff_series_won = 4
    END AS won_championship,

    -- 6. Actual wins minus expected (Pythagorean) wins.
    -- Positive = outperformed scoring-based expectation.
    -- Negative = underperformed scoring-based expectation.
    b.wins - b.pythagorean_wins
        AS wins_above_pythagorean,

    -- 7. Reported team payroll per regular-season victory.
    -- NULLIF prevents division by zero.
    b.team_payroll * 1.0 / NULLIF(b.wins, 0)
        AS payroll_per_win,

    -- 8. Taxable payroll per regular-season victory.
    -- Uses the Spotrac tax-accounting payroll basis.
    b.taxable_payroll * 1.0 / NULLIF(b.wins, 0)
        AS taxable_payroll_per_win,

    -- 9. Total estimated financial cost per regular-season victory.
    -- Includes the source-reported estimated tax component
    -- where present; it is not verified actual ownership spending.
    b.total_estimated_cost * 1.0 / NULLIF(b.wins, 0)
        AS estimated_cost_per_win

FROM a01_v1b AS b
"""

# Register the view in the existing DuckDB connection.
con.execute(sql_a01_v1c).fetchall()


[]

In [23]:

# V1C structural and semantic audit.
# Checks row preservation, key uniqueness, feature coverage,
# and consistency of the derived playoff milestones.

sql_v1c_validation = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

    -- Expected full coverage for competitive outcomes.
    COUNT(games_played) AS n_games_played,
    COUNT(reached_second_round) AS n_second_round,
    COUNT(reached_conference_finals) AS n_conf_finals,
    COUNT(reached_nba_finals) AS n_nba_finals,
    COUNT(won_championship) AS n_championship_flags,
    COUNT(wins_above_pythagorean) AS n_pyth_residuals,

    -- Financial coverage depends on underlying source availability.
    COUNT(payroll_per_win) AS n_payroll_per_win,
    COUNT(taxable_payroll_per_win) AS n_taxable_per_win,
    COUNT(estimated_cost_per_win) AS n_estimated_per_win,

    -- These should all be zero.
    COUNT(*) FILTER (
        WHERE reached_conference_finals = TRUE
          AND reached_second_round IS DISTINCT FROM TRUE
    ) AS conf_without_second,

    COUNT(*) FILTER (
        WHERE reached_nba_finals = TRUE
          AND reached_conference_finals IS DISTINCT FROM TRUE
    ) AS finals_without_conf,

    COUNT(*) FILTER (
        WHERE won_championship = TRUE
          AND reached_nba_finals IS DISTINCT FROM TRUE
    ) AS champions_without_finals,

    COUNT(*) FILTER (
        WHERE payroll_per_win < 0
           OR taxable_payroll_per_win < 0
           OR estimated_cost_per_win < 0
    ) AS negative_cost_ratios

FROM a01_v1c
"""

display(con.sql(sql_v1c_validation).df())


,total_rows,unique_keys,n_games_played,n_second_round,n_conf_finals,n_nba_finals,n_championship_flags,n_pyth_residuals,n_payroll_per_win,n_taxable_per_win,n_estimated_per_win,conf_without_second,finals_without_conf,champions_without_finals,negative_cost_ratios
0,776,776,776,776,776,776,776,776,776,450,450,0,0,0,0


In [24]:

# Count cumulative playoff milestones across all 26 seasons.
# Each Boolean is cast to an integer so TRUE counts as 1.

sql_v1c_milestones = """
SELECT
    SUM(CAST(made_playoffs AS INTEGER))
        AS playoff_appearances,

    SUM(CAST(reached_second_round AS INTEGER))
        AS second_round_appearances,

    SUM(CAST(reached_conference_finals AS INTEGER))
        AS conference_finals_appearances,

    SUM(CAST(reached_nba_finals AS INTEGER))
        AS nba_finals_appearances,

    SUM(CAST(won_championship AS INTEGER))
        AS championships

FROM a01_v1c
"""

display(con.sql(sql_v1c_milestones).df())


,playoff_appearances,second_round_appearances,conference_finals_appearances,nba_finals_appearances,championships
0,416.0,208.0,104.0,52.0,26.0


A01 v1 — Methodology Checkpoint

Grain: One franchise-season, keyed by franchise_id and season.
Sources: C04 team seasons, C01 league season thresholds, and C10 team apron seasons.
Feature engineering: 20 derived features across financial normalization (5), financial threshold exposure (6), and competitive outcomes/spending efficiency (9).
Validation: 776 rows, 776 unique keys; agreed structural and consistency checks passed.
Coverage: Taxable payroll and estimated cost measures cover 450 franchise-seasons; reported tax-paid ratios cover 268; second-apron features cover 90 applicable observations.
Accounting limitations: Payroll, taxable payroll, apron allocations, and estimated cost are not interchangeable. Missing estimated tax is not proof of zero liability.
Competitive limitations: Spending per win is descriptive rather than causal. Season lengths differ historically. Pythagorean residuals describe deviations from scoring-based expectations, not necessarily managerial skill.
Persistence: A01 currently exists as a temporary DuckDB view and has not yet been exported.

In [25]:

# A01 is our current analytical dataset.
# This view points to the latest completed version.
# It does not duplicate data or create a permanent file.

sql_a01_current = """
CREATE OR REPLACE TEMP VIEW a01 AS
SELECT *
FROM a01_v1c
"""

con.execute(sql_a01_current).fetchall()

# Quick confirmation that the alias resolves correctly.
display(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys
    FROM a01
""").df())


,total_rows,unique_keys
0,776,776


In [26]:

# A01 v2A — Group A: Financial Position
# Audit the number of franchises with usable values
# for each financial measure, within each season.

sql_v2a_financial_coverage = """
SELECT
    season,

    -- Number of franchise-season observations
    COUNT(*) AS n_teams,

    -- Number of franchises with each financial measure
    COUNT(team_payroll) AS n_payroll,
    COUNT(salary_cap_hit) AS n_cap_hit,
    COUNT(taxable_payroll) AS n_taxable,
    COUNT(total_apron_allocations) AS n_apron

FROM a01_v1c
GROUP BY season
ORDER BY season
"""

display(con.sql(sql_v2a_financial_coverage).df())


,season,n_teams,n_payroll,n_cap_hit,n_taxable,n_apron
0,2000-2001,29,29,29,0,0
1,2001-2002,29,29,29,0,0
2,2002-2003,29,29,29,0,0
3,2003-2004,29,29,29,0,0
4,2004-2005,30,30,30,0,0
5,2005-2006,30,30,30,0,0
6,2006-2007,30,30,30,0,0
7,2007-2008,30,30,30,0,0
8,2008-2009,30,30,30,0,0
9,2009-2010,30,30,30,0,0


In [27]:

# A01 v2A — Group A: Financial Position
# Generate 8 comparable features for each of 3 financial measures.
# Add 1 separate salary-cap-hit rank: 8 * 3 + 1 = 25.

financial_measures = {
    "team_payroll": "payroll",
    "taxable_payroll": "taxable_payroll",
    "total_apron_allocations": "apron_allocations"
}

feature_expressions = []

for column, prefix in financial_measures.items():

    # Rank franchises by spending within each season.
    # Highest spending receives rank 1; ties share the same rank.
    # CASE preserves NULL when financial data is unavailable,
    # preventing historical missing values from receiving false ranks.
    feature_expressions.append(f"""
        CASE WHEN {column} IS NOT NULL THEN
            RANK() OVER (
                PARTITION BY season
                ORDER BY {column} DESC NULLS LAST
            )
        END AS {prefix}_rank
    """)

    # Ascending PERCENT_RANK makes the highest value approach 1.
    # Missing financial observations are excluded from the ranking
    # calculation and receive NULL.
    feature_expressions.append(f"""
        CASE WHEN {column} IS NOT NULL THEN
            PERCENT_RANK() OVER (
                PARTITION BY season, ({column} IS NOT NULL)
                ORDER BY {column} ASC NULLS LAST
            )
        END AS {prefix}_percentile
    """)

    # Compare each franchise's spending to the season's median.
    # NULLIF prevents division by zero.
    feature_expressions.append(f"""
        {column} * 1.0 /
        NULLIF(
            MEDIAN({column}) OVER (PARTITION BY season),
            0
        ) AS {prefix}_to_league_median
    """)

    # Absolute spending difference from the season's median.
    feature_expressions.append(f"""
        {column} -
        MEDIAN({column}) OVER (PARTITION BY season)
        AS {prefix}_vs_league_median
    """)

    # Compare each franchise's spending to the season's mean.
    feature_expressions.append(f"""
        {column} * 1.0 /
        NULLIF(
            AVG({column}) OVER (PARTITION BY season),
            0
        ) AS {prefix}_to_league_mean
    """)

    # Absolute spending difference from the season's mean.
    feature_expressions.append(f"""
        {column} -
        AVG({column}) OVER (PARTITION BY season)
        AS {prefix}_vs_league_mean
    """)

    # Divide each season's valid financial observations into
    # four spending tiers, with Q1 representing highest spending.
    #
    # Partitioning by availability prevents missing observations
    # from affecting quartile assignments for valid observations.
    feature_expressions.append(f"""
        CASE WHEN {column} IS NOT NULL THEN
            NTILE(4) OVER (
                PARTITION BY season, ({column} IS NOT NULL)
                ORDER BY {column} DESC NULLS LAST, franchise_id
            )
        END AS {prefix}_quartile
    """)

    # Calculate each franchise's share of league-wide spending
    # under the corresponding financial accounting definition.
    feature_expressions.append(f"""
        {column} * 1.0 /
        NULLIF(
            SUM({column}) OVER (PARTITION BY season),
            0
        ) AS {prefix}_share_of_league_total
    """)

# Rank franchises separately by salary-cap hit.
# This is the 25th feature in Group A.
feature_expressions.append("""
    RANK() OVER (
        PARTITION BY season
        ORDER BY salary_cap_hit DESC NULLS LAST
    ) AS salary_cap_hit_rank
""")

# Confirm that exactly 25 expressions were generated.
assert len(feature_expressions) == 25

# Combine the expressions into a single SQL SELECT statement.
feature_sql = ",\n".join(feature_expressions)

# Build the financial-position view on top of v1C.
# b.* retains all previously engineered features.
sql_a01_v2a_financial = f"""
CREATE OR REPLACE TEMP VIEW a01_v2a_financial AS
SELECT
    b.*,
    {feature_sql}
FROM a01_v1c AS b
"""

# Execute the SQL to create or replace the temporary view.
con.execute(sql_a01_v2a_financial).fetchall()

print("Created a01_v2a_financial with 25 new features.")


Created a01_v2a_financial with 25 new features.


In [28]:

# Check that Group A preserves the A01 grain and coverage.

sql_v2a_financial_audit = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

    COUNT(payroll_rank) AS n_payroll_rank,
    COUNT(payroll_percentile) AS n_payroll_percentile,
    COUNT(payroll_quartile) AS n_payroll_quartile,

    COUNT(taxable_payroll_rank) AS n_taxable_rank,
    COUNT(taxable_payroll_percentile) AS n_taxable_percentile,
    COUNT(taxable_payroll_quartile) AS n_taxable_quartile,

    COUNT(apron_allocations_rank) AS n_apron_rank,
    COUNT(apron_allocations_percentile) AS n_apron_percentile,
    COUNT(apron_allocations_quartile) AS n_apron_quartile,

    COUNT(salary_cap_hit_rank) AS n_cap_hit_rank

FROM a01_v2a_financial
"""

display(con.sql(sql_v2a_financial_audit).df())


,total_rows,unique_keys,n_payroll_rank,n_payroll_percentile,n_payroll_quartile,n_taxable_rank,n_taxable_percentile,n_taxable_quartile,n_apron_rank,n_apron_percentile,n_apron_quartile,n_cap_hit_rank
0,776,776,776,776,776,450,450,450,450,450,450,776


In [29]:

# Validate the three financial-position feature families.
# Expected: 0 invalid rows; league shares sum to 1 for covered seasons.

audit_parts = []

for column, prefix in financial_measures.items():

    audit_parts.append(f"""
        SELECT
            '{prefix}' AS measure,
            COUNT(*) AS available_rows,

            -- Check ranks, percentiles, and quartiles
            COUNT(*) FILTER (
                WHERE {prefix}_rank < 1
                   OR {prefix}_rank > n_teams
                   OR {prefix}_percentile NOT BETWEEN 0 AND 1
                   OR {prefix}_quartile NOT BETWEEN 1 AND 4
            ) AS invalid_positions,

            -- Missing source values should produce missing features
            COUNT(*) FILTER (
                WHERE {column} IS NULL
                  AND (
                      {prefix}_rank IS NOT NULL
                      OR {prefix}_percentile IS NOT NULL
                      OR {prefix}_quartile IS NOT NULL
                      OR {prefix}_share_of_league_total IS NOT NULL
                  )
            ) AS invalid_missingness,

            -- Each covered season should sum to 100% of league total
            COUNT(*) FILTER (
                WHERE ABS(league_share - 1.0) > 0.000001
            ) AS invalid_share_seasons

        FROM (
            SELECT
                *,
                COUNT(*) OVER (PARTITION BY season) AS n_teams,
                SUM({prefix}_share_of_league_total)
                    OVER (PARTITION BY season) AS league_share
            FROM a01_v2a_financial
        ) AS checks
        WHERE {column} IS NOT NULL
    """)

audit_sql = "\nUNION ALL\n".join(audit_parts)

display(con.sql(audit_sql).df())


,measure,available_rows,invalid_positions,invalid_missingness,invalid_share_seasons
0,payroll,776,0,0,0
1,taxable_payroll,450,0,0,0
2,apron_allocations,450,0,0,0


In [30]:

# Compare the three financial accounting measures and their rankings.
# Dollar amounts are displayed in millions for readability.

display(con.sql("""
    SELECT
        team_name,

        ROUND(team_payroll / 1e6, 1) AS payroll_m,
        payroll_rank,
        ROUND(payroll_percentile * 100, 1) AS payroll_pct,

        ROUND(taxable_payroll / 1e6, 1) AS taxable_m,
        taxable_payroll_rank,

        ROUND(total_apron_allocations / 1e6, 1) AS apron_m,
        apron_allocations_rank

    FROM a01_v2a_financial
    WHERE season = '2023-2024'
    ORDER BY payroll_rank, team_name
    LIMIT 10
""").df())


,team_name,payroll_m,payroll_rank,payroll_pct,taxable_m,taxable_payroll_rank,apron_m,apron_allocations_rank
0,Golden State Warriors,209.4,1,100.0,205.6,1,207.8,1
1,Los Angeles Clippers,201.4,2,96.6,200.1,2,200.1,2
2,Phoenix Suns,193.8,3,93.1,191.3,3,192.2,3
3,Milwaukee Bucks,187.3,4,89.7,182.2,5,184.6,5
4,Boston Celtics,186.9,5,86.2,184.9,4,188.6,4
5,Denver Nuggets,180.9,6,82.8,176.9,6,178.1,6
6,Miami Heat,177.1,7,79.3,174.7,7,177.2,7
7,Los Angeles Lakers,169.9,8,75.9,168.1,8,169.0,8
8,Dallas Mavericks,167.8,9,72.4,164.6,10,166.1,11
9,New Orleans Pelicans,167.4,10,69.0,165.0,9,164.9,15


In [31]:

# Pre-2011 financial-position features should remain NULL.

display(con.sql("""
    SELECT
        COUNT(*) AS pre_2011_rows,

        COUNT(taxable_payroll_rank)
            AS unexpected_taxable_ranks,

        COUNT(apron_allocations_rank)
            AS unexpected_apron_ranks,

        COUNT(taxable_payroll_share_of_league_total)
            AS unexpected_taxable_shares,

        COUNT(apron_allocations_share_of_league_total)
            AS unexpected_apron_shares

    FROM a01_v2a_financial
    WHERE season < '2011-2012'
""").df())


,pre_2011_rows,unexpected_taxable_ranks,unexpected_apron_ranks,unexpected_taxable_shares,unexpected_apron_shares
0,326,0,0,0,0


In [32]:

# A01 v2A Group A — inspect the 25 newly engineered features.
# The last 25 columns are the features we just created.

df_financial = con.sql("""
    SELECT *
    FROM a01_v2a_financial
""").df()

# Confirm the dataset's dimensions.
print("Total rows:", len(df_financial))
print("Total columns:", len(df_financial.columns))

# List the 25 new feature names.
new_features = df_financial.columns[-25:]

print("\nNew Group A features:")
for feature in new_features:
    print(feature)

# Display one franchise-season with all 25 features.
display(
    df_financial.loc[
        df_financial["season"] == "2023-2024",
        ["team_name", "season"] + list(new_features)
    ].head(1).T
)


Total rows: 776
Total columns: 118

New Group A features:
payroll_rank
payroll_percentile
payroll_to_league_median
payroll_vs_league_median
payroll_to_league_mean
payroll_vs_league_mean
payroll_quartile
payroll_share_of_league_total
taxable_payroll_rank
taxable_payroll_percentile
taxable_payroll_to_league_median
taxable_payroll_vs_league_median
taxable_payroll_to_league_mean
taxable_payroll_vs_league_mean
taxable_payroll_quartile
taxable_payroll_share_of_league_total
apron_allocations_rank
apron_allocations_percentile
apron_allocations_to_league_median
apron_allocations_vs_league_median
apron_allocations_to_league_mean
apron_allocations_vs_league_mean
apron_allocations_quartile
apron_allocations_share_of_league_total
salary_cap_hit_rank


,746
team_name,Golden State Warriors
season,2023-2024
payroll_rank,1
payroll_percentile,1.0
payroll_to_league_median,1.267839
payroll_vs_league_median,44227481.5
payroll_to_league_mean,1.273561
payroll_vs_league_mean,44969359.633333
payroll_quartile,1
payroll_share_of_league_total,0.042452


In [33]:

# Compare financial positions for all 30 franchises in 2023-2024.
# Dollar amounts are converted to millions for display.

display(con.sql("""
    SELECT
        team_name,
        ROUND(team_payroll / 1e6, 1) AS payroll_m,
        payroll_rank,
        ROUND(payroll_percentile * 100, 1) AS payroll_percentile_pct,
        ROUND(payroll_to_league_mean, 2) AS payroll_vs_mean_ratio,
        payroll_quartile,
        ROUND(payroll_share_of_league_total * 100, 2)
            AS league_payroll_share_pct
    FROM a01_v2a_financial
    WHERE season = '2023-2024'
    ORDER BY payroll_rank, team_name
""").df())


,team_name,payroll_m,payroll_rank,payroll_percentile_pct,payroll_vs_mean_ratio,payroll_quartile,league_payroll_share_pct
0,Golden State Warriors,209.4,1,100.0,1.27,1,4.25
1,Los Angeles Clippers,201.4,2,96.6,1.22,1,4.08
2,Phoenix Suns,193.8,3,93.1,1.18,1,3.93
3,Milwaukee Bucks,187.3,4,89.7,1.14,1,3.80
4,Boston Celtics,186.9,5,86.2,1.14,1,3.79
5,Denver Nuggets,180.9,6,82.8,1.10,1,3.67
6,Miami Heat,177.1,7,79.3,1.08,1,3.59
7,Los Angeles Lakers,169.9,8,75.9,1.03,1,3.44
8,Dallas Mavericks,167.8,9,72.4,1.02,2,3.40
9,New Orleans Pelicans,167.4,10,69.0,1.02,2,3.39


In [34]:

# A01 v2A Group B — compact source check
# Confirm competitive data coverage and inspect playoff scoring.

display(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(wins) AS n_wins,
        COUNT(losses) AS n_losses,
        COUNT(win_pct) AS n_win_pct,
        COUNT(round_score) AS n_round_score,
        MIN(round_score) AS min_round_score,
        MAX(round_score) AS max_round_score
    FROM a01_v2a_financial
""").df())

# Inspect how round_score corresponds to playoff outcomes.
display(con.sql("""
    SELECT
        round_score,
        COUNT(*) AS franchise_seasons,
        SUM(CASE WHEN made_playoffs THEN 1 ELSE 0 END)
            AS playoff_appearances,
        SUM(CASE WHEN won_championship THEN 1 ELSE 0 END)
            AS championships
    FROM a01_v2a_financial
    GROUP BY round_score
    ORDER BY round_score
""").df())


,total_rows,n_wins,n_losses,n_win_pct,n_round_score,min_round_score,max_round_score
0,776,776,776,776,416,1.0,4.0


,round_score,franchise_seasons,playoff_appearances,championships
0,1.0,208,208.0,0.0
1,2.0,104,104.0,0.0
2,3.0,52,52.0,0.0
3,4.0,52,52.0,26.0
4,NaN,360,0.0,0.0


In [35]:

# A01 v2A — Group B: Competitive Position
# Inherit all existing features and append 6 new ones.

sql_a01_v2a_competitive = """
CREATE OR REPLACE TEMP VIEW a01_v2a_competitive AS

SELECT
    b.*,

    -- 1. League-wide regular-season rank.
    -- Highest winning percentage receives rank 1.
    RANK() OVER (
        PARTITION BY season
        ORDER BY win_pct DESC
    ) AS win_pct_rank,

    -- 2. Season-relative competitive quartile.
    -- Q1 = strongest; Q4 = weakest.
    -- franchise_id breaks ties deterministically.
    NTILE(4) OVER (
        PARTITION BY season
        ORDER BY win_pct DESC, franchise_id
    ) AS win_pct_quartile,

    -- 3. Conventional NBA games above/below .500.
    -- Example: 50-32 produces +18.
    wins - losses AS games_above_500,

    -- 4. Wins relative to the season's league mean.
    wins - AVG(wins) OVER (
        PARTITION BY season
    ) AS wins_vs_league_mean,

    -- 5. Wins relative to the season's league median.
    wins - MEDIAN(wins) OVER (
        PARTITION BY season
    ) AS wins_vs_league_median,

    -- 6. Postseason advancement rank.
    -- Champions receive a higher score than Finals runners-up.
    -- Non-playoff teams receive a score of -1 for ranking only.
    RANK() OVER (
        PARTITION BY season
        ORDER BY
            CASE
                WHEN won_championship THEN 5
                ELSE COALESCE(round_score, -1)
            END DESC
    ) AS playoff_advancement_rank

FROM a01_v2a_financial AS b
"""

con.execute(sql_a01_v2a_competitive).fetchall()

print("Created a01_v2a_competitive with 6 new features.")


Created a01_v2a_competitive with 6 new features.


In [36]:

# Compact structural and semantic validation for Group B.

display(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

        COUNT(win_pct_rank) AS n_win_pct_rank,
        COUNT(win_pct_quartile) AS n_quartile,
        COUNT(games_above_500) AS n_games_above_500,
        COUNT(wins_vs_league_mean) AS n_mean_gap,
        COUNT(wins_vs_league_median) AS n_median_gap,
        COUNT(playoff_advancement_rank) AS n_playoff_rank,

        COUNT(*) FILTER (
            WHERE win_pct_quartile NOT BETWEEN 1 AND 4
               OR win_pct_rank < 1
               OR playoff_advancement_rank < 1
        ) AS invalid_ranks,

        COUNT(*) FILTER (
            WHERE won_championship
              AND playoff_advancement_rank <> 1
        ) AS invalid_champion_ranks,

        COUNT(*) FILTER (
            WHERE games_above_500 <> wins - losses
        ) AS invalid_500_gaps

    FROM a01_v2a_competitive
""").df())


,total_rows,unique_keys,n_win_pct_rank,n_quartile,n_games_above_500,n_mean_gap,n_median_gap,n_playoff_rank,invalid_ranks,invalid_champion_ranks,invalid_500_gaps
0,776,776,776,776,776,776,776,776,0,0,0


In [37]:

# Inspect the 2023-2024 competitive positions.
# All 30 teams are included; no LIMIT is applied.

display(con.sql("""
    SELECT
        team_name,
        wins,
        losses,
        ROUND(win_pct, 3) AS win_pct,
        win_pct_rank,
        win_pct_quartile,
        games_above_500,
        ROUND(wins_vs_league_mean, 1) AS wins_vs_mean,
        ROUND(wins_vs_league_median, 1) AS wins_vs_median,
        round_score,
        playoff_advancement_rank

    FROM a01_v2a_competitive
    WHERE season = '2023-2024'
    ORDER BY win_pct_rank, team_name
""").df())


,team_name,wins,losses,win_pct,win_pct_rank,win_pct_quartile,games_above_500,wins_vs_mean,wins_vs_median,round_score,playoff_advancement_rank
0,Boston Celtics,64,18,0.780,1,1,46,23.0,17.5,4.0,1
1,Denver Nuggets,57,25,0.695,2,1,32,16.0,10.5,2.0,5
2,Oklahoma City Thunder,57,25,0.695,2,1,32,16.0,10.5,2.0,5
3,Minnesota Timberwolves,56,26,0.683,4,1,30,15.0,9.5,3.0,3
4,Los Angeles Clippers,51,31,0.622,5,1,20,10.0,4.5,1.0,9
5,Dallas Mavericks,50,32,0.610,6,1,18,9.0,3.5,4.0,2
6,New York Knicks,50,32,0.610,6,1,18,9.0,3.5,2.0,5
7,Milwaukee Bucks,49,33,0.598,8,2,16,8.0,2.5,1.0,9
8,New Orleans Pelicans,49,33,0.598,8,1,16,8.0,2.5,1.0,9
9,Phoenix Suns,49,33,0.598,8,2,16,8.0,2.5,1.0,9


In [38]:

# A01 v2A — Group C: Spending Efficiency
# Generate 4 efficiency features for each of 3 financial measures.
# 4 * 3 = 12 new features.

# Map each feature prefix to its canonical financial column.
financial_measures = {
    "payroll": "team_payroll",
    "taxable_payroll": "taxable_payroll",
    "apron_allocations": "total_apron_allocations"
}

feature_expressions = []

for prefix, column in financial_measures.items():

    # 1. Financial rank minus competitive rank.
    # Positive = better competitive rank than spending rank.
    # Example: spending rank 15, win rank 5 -> gap +10.
    feature_expressions.append(f"""
        {prefix}_rank - win_pct_rank
        AS {prefix}_rank_win_pct_gap
    """)

    # 2. Financial quartile minus competitive quartile.
    # Q1 = highest spending or strongest performance.
    # Positive = outperforming the franchise's spending tier.
    feature_expressions.append(f"""
        {prefix}_quartile - win_pct_quartile
        AS {prefix}_quartile_win_pct_gap
    """)

    # 3. Normalized spending efficiency.
    #
    # Formula:
    # (Team win_pct / season mean win_pct)
    # divided by
    # (Team spending / season median spending)
    #
    # > 1 = greater competitive output relative to spending.
    # < 1 = lower competitive output relative to spending.
    #
    # This is a descriptive ratio, not a causal ROI estimate.
    # NULLIF protects against division by zero.
    feature_expressions.append(f"""
        (
            win_pct /
            NULLIF(
                AVG(win_pct) OVER (PARTITION BY season),
                0
            )
        )
        /
        NULLIF(
            {column} /
            NULLIF(
                MEDIAN({column}) OVER (PARTITION BY season),
                0
            ),
            0
        )
        AS {prefix}_normalized_efficiency
    """)

    # 4. Winning percentage relative to teams in the same
    # season and spending quartile.
    #
    # The peer mean includes the focal franchise itself.
    # Positive = outperforming the spending-quartile average.
    # Missing financial quartiles produce NULL.
    feature_expressions.append(f"""
        CASE
            WHEN {prefix}_quartile IS NOT NULL
            THEN win_pct - AVG(win_pct) OVER (
                PARTITION BY season, {prefix}_quartile
            )
        END AS win_pct_vs_{prefix}_quartile_mean
    """)

# Verify exactly 12 features were generated.
assert len(feature_expressions) == 12

# Combine the expressions into one SQL SELECT statement.
feature_sql = ",\n".join(feature_expressions)

# Build Group C on top of the existing Group B view.
# b.* preserves all previously engineered features.
sql_a01_v2a_efficiency = f"""
CREATE OR REPLACE TEMP VIEW a01_v2a_efficiency AS
SELECT
    b.*,
    {feature_sql}
FROM a01_v2a_competitive AS b
"""

# Create the temporary view.
con.execute(sql_a01_v2a_efficiency).fetchall()

print("Created a01_v2a_efficiency with 12 new features.")


Created a01_v2a_efficiency with 12 new features.


In [39]:

# A01 v2A — Group C: Consolidated Audit
# Check row preservation, feature coverage, and arithmetic accuracy.

# List all 12 Group C features.
group_c_features = [
    f"{prefix}_{suffix}"
    for prefix in financial_measures
    for suffix in [
        "rank_win_pct_gap",
        "quartile_win_pct_gap",
        "normalized_efficiency"
    ]
] + [
    f"win_pct_vs_{prefix}_quartile_mean"
    for prefix in financial_measures
]

assert len(group_c_features) == 12

# Count non-NULL observations for each feature.
coverage_sql = ",\n".join(
    f'COUNT("{feature}") AS "{feature}"'
    for feature in group_c_features
)

coverage = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys,
        {coverage_sql}
    FROM a01_v2a_efficiency
""").df()

print("GROUP C — ROW COUNTS AND FEATURE COVERAGE")
display(coverage.T.rename(columns={0: "non_null_count"}))


# Validate financial rank and quartile gap calculations.
# IS DISTINCT FROM treats NULL comparisons explicitly,
# so unexpected missing values are detectable.
checks = []

for prefix in financial_measures:

    checks.extend([
        f"""
        COUNT(*) FILTER (
            WHERE {prefix}_rank_win_pct_gap
                  IS DISTINCT FROM
                  ({prefix}_rank - win_pct_rank)
        ) AS {prefix}_rank_gap_errors
        """,

        f"""
        COUNT(*) FILTER (
            WHERE {prefix}_quartile_win_pct_gap
                  IS DISTINCT FROM
                  ({prefix}_quartile - win_pct_quartile)
        ) AS {prefix}_quartile_gap_errors
        """
    ])

audit_sql = f"""
SELECT
    {",".join(checks)}
FROM a01_v2a_efficiency
"""

print("\nGROUP C — ARITHMETIC VALIDATION")
display(con.sql(audit_sql).df().T)


GROUP C — ROW COUNTS AND FEATURE COVERAGE


,non_null_count
total_rows,776
unique_keys,776
payroll_rank_win_pct_gap,776
payroll_quartile_win_pct_gap,776
payroll_normalized_efficiency,776
taxable_payroll_rank_win_pct_gap,450
taxable_payroll_quartile_win_pct_gap,450
taxable_payroll_normalized_efficiency,450
apron_allocations_rank_win_pct_gap,450
apron_allocations_quartile_win_pct_gap,450



GROUP C — ARITHMETIC VALIDATION


,0
payroll_rank_gap_errors,0
payroll_quartile_gap_errors,0
taxable_payroll_rank_gap_errors,0
taxable_payroll_quartile_gap_errors,0
apron_allocations_rank_gap_errors,0
apron_allocations_quartile_gap_errors,0


In [40]:

# A01 v2A — Group D: Source and Feature Audit
# Read-only diagnostics before implementing financial exposure.
# This cell does not create or replace any views.

import pandas as pd

# 1. Identify financial, threshold, and regulatory fields
# already available in the latest A01 view.
columns = con.sql("""
    DESCRIBE SELECT * FROM a01_v2a_efficiency
""").df()

keywords = [
    "tax", "apron", "threshold", "hard_cap",
    "estimated", "cost", "payroll", "space"
]

relevant_columns = columns[
    columns["column_name"].str.lower().apply(
        lambda name: any(word in name for word in keywords)
    )
]

print("1. EXISTING FINANCIAL / EXPOSURE COLUMNS")
display(relevant_columns[["column_name", "column_type"]])


# 2. Inspect coverage and distributions of key source fields.
# GROUP BY season helps distinguish historical coverage gaps.
print("\n2. TAX AND COST SOURCE COVERAGE BY SEASON")

coverage = con.sql("""
    SELECT
        season,
        COUNT(*) AS teams,
        COUNT(taxable_payroll) AS taxable_payroll_n,
        COUNT(tax_paid) AS tax_paid_n,
        COUNT(*) FILTER (
            WHERE tax_paid = 0
        ) AS explicit_zero_tax_n,
        COUNT(*) FILTER (
            WHERE tax_paid > 0
        ) AS positive_tax_n,
        COUNT(estimated_tax_bill) AS estimated_bill_n,
        COUNT(total_estimated_cost) AS total_cost_n,
        COUNT(total_apron_allocations) AS apron_n,
        SUM(tax_paid) AS observed_tax_total,
        MEDIAN(tax_paid) AS observed_tax_median
    FROM a01_v2a_efficiency
    GROUP BY season
    ORDER BY season
""").df()

display(coverage)


# 3. Investigate how total_estimated_cost relates to
# payroll, actual tax paid, and estimated tax liability.
# These are diagnostic comparisons, not assumed definitions.
print("\n3. TOTAL ESTIMATED COST — CONSTRUCTION CHECK")

cost_check = con.sql("""
    SELECT
        COUNT(total_estimated_cost) AS total_cost_n,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND team_payroll IS NOT NULL
              AND ABS(total_estimated_cost - team_payroll) < 1
        ) AS equals_payroll_n,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND tax_paid IS NOT NULL
              AND ABS(
                  total_estimated_cost
                  - (team_payroll + tax_paid)
              ) < 1
        ) AS equals_payroll_plus_actual_tax_n,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND estimated_tax_bill IS NOT NULL
              AND ABS(
                  total_estimated_cost
                  - (team_payroll + estimated_tax_bill)
              ) < 1
        ) AS equals_payroll_plus_estimated_tax_n

    FROM a01_v2a_efficiency
""").df()

display(cost_check.T.rename(columns={0: "count"}))


# 4. Discover whether S19 or hard-cap data has been
# registered as a table or view in DuckDB.
print("\n4. S19 / HARD-CAP SOURCE DISCOVERY")

source_tables = con.sql("""
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE lower(table_name) LIKE '%s19%'
       OR lower(table_name) LIKE '%hard%'
       OR lower(table_name) LIKE '%transaction%'
    ORDER BY table_schema, table_name
""").df()

display(source_tables)

print("\nGroup D diagnostic audit complete.")


1. EXISTING FINANCIAL / EXPOSURE COLUMNS


,column_name,column_type
39,team_payroll,BIGINT
41,salary_cap_space,DOUBLE
43,s18_tax_threshold,DOUBLE
44,taxable_payroll,DOUBLE
45,tax_space,DOUBLE
...,...,...
131,win_pct_vs_taxable_payroll_quartile_mean,DOUBLE
132,apron_allocations_rank_win_pct_gap,BIGINT
133,apron_allocations_quartile_win_pct_gap,BIGINT
134,apron_allocations_normalized_efficiency,DOUBLE



2. TAX AND COST SOURCE COVERAGE BY SEASON


,season,teams,taxable_payroll_n,tax_paid_n,explicit_zero_tax_n,positive_tax_n,estimated_bill_n,total_cost_n,apron_n,observed_tax_total,observed_tax_median
0,2000-2001,29,0,0,0,0,0,0,0,NaN,NaN
1,2001-2002,29,0,0,0,0,0,0,0,NaN,NaN
2,2002-2003,29,0,29,13,16,0,0,0,173320000.0,890000.0
3,2003-2004,29,0,29,18,11,0,0,0,154450000.0,0.0
4,2004-2005,30,0,30,29,1,0,0,0,760000.0,0.0
5,2005-2006,30,0,30,24,6,0,0,0,71640000.0,0.0
6,2006-2007,30,0,30,25,5,0,0,0,55560000.0,0.0
7,2007-2008,30,0,30,22,8,0,0,0,92450000.0,0.0
8,2008-2009,30,0,30,23,7,0,0,0,87360000.0,0.0
9,2009-2010,30,0,30,19,11,0,0,0,111080000.0,0.0



3. TOTAL ESTIMATED COST — CONSTRUCTION CHECK


,count
total_cost_n,450
equals_payroll_n,0
equals_payroll_plus_actual_tax_n,0
equals_payroll_plus_estimated_tax_n,0



4. S19 / HARD-CAP SOURCE DISCOVERY


,table_schema,table_name,table_type



Group D diagnostic audit complete.


In [41]:

# A01 v2A — Group D: Targeted Follow-Up
# Inspect existing threshold features and total-cost construction.
# Read-only: does not modify existing views.

# 1. Show all existing columns related to threshold
# distance, tax exposure, and apron classification.
print("EXISTING THRESHOLD / EXPOSURE FEATURES")

for name in columns["column_name"]:
    if any(term in name.lower() for term in [
        "threshold", "apron", "tax_space",
        "tax_", "hard_cap", "cost"
    ]):
        print(name)


# 2. Show representative financial records to investigate
# how total_estimated_cost relates to other accounting fields.
print("\nTOTAL ESTIMATED COST — SAMPLE RECORDS")

display(con.sql("""
    SELECT
        season,
        franchise_id,
        team_payroll,
        taxable_payroll,
        total_apron_allocations,
        tax_paid,
        estimated_tax_bill,
        total_estimated_cost,
        total_estimated_cost - team_payroll
            AS cost_minus_payroll,
        total_estimated_cost - taxable_payroll
            AS cost_minus_taxable_payroll
    FROM a01_v2a_efficiency
    WHERE total_estimated_cost IS NOT NULL
    ORDER BY season DESC, franchise_id
    LIMIT 12
""").df())


EXISTING THRESHOLD / EXPOSURE FEATURES
s18_tax_threshold
tax_space
tax_paid
estimated_tax_bill
total_estimated_cost
has_luxury_tax_source
league_luxury_tax_threshold
league_first_apron
league_second_apron
total_apron_allocations
first_apron_threshold
second_apron_threshold
first_apron_space
second_apron_space
above_first_apron
above_second_apron
first_apron_hard_capped
second_apron_hard_capped
second_apron_applicable
apron_verification_status
apron_source_coverage
taxable_payroll_to_tax_ratio
first_apron_allocation_ratio
second_apron_allocation_ratio
tax_threshold_distance
first_apron_distance
second_apron_distance
above_tax_threshold
tax_paid_to_payroll_ratio
estimated_cost_to_taxable_ratio
estimated_cost_per_win
apron_allocations_rank
apron_allocations_percentile
apron_allocations_to_league_median
apron_allocations_vs_league_median
apron_allocations_to_league_mean
apron_allocations_vs_league_mean
apron_allocations_quartile
apron_allocations_share_of_league_total
apron_allocations_ran

,season,franchise_id,team_payroll,taxable_payroll,total_apron_allocations,tax_paid,estimated_tax_bill,total_estimated_cost,cost_minus_payroll,cost_minus_taxable_payroll
0,2025-2026,1610612737,177744274,186614541.0,186614541,NaN,NaN,186614541.0,8870267.0,0.0
1,2025-2026,1610612738,187492509,187885254.0,187885254,NaN,NaN,187885254.0,392745.0,0.0
2,2025-2026,1610612739,211858381,212491192.0,212404326,NaN,69437508.0,281928700.0,70070319.0,69437508.0
3,2025-2026,1610612740,190898076,187727672.0,193967891,NaN,NaN,187727672.0,-3170404.0,0.0
4,2025-2026,1610612741,187139097,183086419.0,185036418,NaN,NaN,183086419.0,-4052678.0,0.0
5,2025-2026,1610612742,185068401,186542141.0,187783308,NaN,NaN,186542141.0,1473740.0,0.0
6,2025-2026,1610612743,191600929,187289504.0,192009140,NaN,NaN,187289504.0,-4311425.0,0.0
7,2025-2026,1610612744,205124954,205114503.0,205577466,NaN,67909145.0,273023648.0,67898694.0,67909145.0
8,2025-2026,1610612745,199783669,194771442.0,194771442,NaN,7174303.0,201945745.0,2162076.0,7174303.0
9,2025-2026,1610612746,191938984,190116914.0,190116914,NaN,6665742.0,196782656.0,4843672.0,6665742.0


In [42]:

# A01 v2A — Group D: Final Definition Checks
# Read-only checks for cost construction and threshold formulas.

print("1. TOTAL ESTIMATED COST FORMULA")

display(con.sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
        ) AS total_cost_rows,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND ABS(
                  total_estimated_cost
                  - (
                      taxable_payroll
                      + COALESCE(estimated_tax_bill, 0)
                  )
              ) > 1
        ) AS cost_formula_mismatches,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND estimated_tax_bill IS NULL
        ) AS cost_rows_without_estimated_bill

    FROM a01_v2a_efficiency
""").df())


print("\n2. THRESHOLD FORMULA CHECKS")

display(con.sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE tax_threshold_distance IS NOT NULL
              AND taxable_payroll IS NOT NULL
              AND league_luxury_tax_threshold IS NOT NULL
              AND ABS(
                  tax_threshold_distance
                  - (
                      taxable_payroll
                      - league_luxury_tax_threshold
                  )
              ) > 1
        ) AS tax_distance_mismatches,

        COUNT(*) FILTER (
            WHERE first_apron_distance IS NOT NULL
              AND total_apron_allocations IS NOT NULL
              AND first_apron_threshold IS NOT NULL
              AND ABS(
                  first_apron_distance
                  - (
                      total_apron_allocations
                      - first_apron_threshold
                  )
              ) > 1
        ) AS first_apron_distance_mismatches,

        COUNT(*) FILTER (
            WHERE second_apron_distance IS NOT NULL
              AND total_apron_allocations IS NOT NULL
              AND second_apron_threshold IS NOT NULL
              AND ABS(
                  second_apron_distance
                  - (
                      total_apron_allocations
                      - second_apron_threshold
                  )
              ) > 1
        ) AS second_apron_distance_mismatches

    FROM a01_v2a_efficiency
""").df())


1. TOTAL ESTIMATED COST FORMULA


,total_cost_rows,cost_formula_mismatches,cost_rows_without_estimated_bill
0,450,0,356



2. THRESHOLD FORMULA CHECKS


,tax_distance_mismatches,first_apron_distance_mismatches,second_apron_distance_mismatches
0,0,0,0


In [43]:

# A01 v2A — Group D: Financial Exposure
# Creates 27 new features without modifying canonical data.
#
# Input:  a01_v2a_efficiency
# Output: a01_v2a_exposure

# Use a list so we can count and inspect every new feature.
d_features = []

def add_feature(expression, name):
    # Each expression becomes one named SQL column.
    d_features.append(f"{expression} AS {name}")


# ---------------------------------------------------------
# D1 — LUXURY-TAX BURDEN
# ---------------------------------------------------------

# Actual tax / taxable payroll is intentionally omitted:
# these fields have no overlapping non-null observations.
# Existing tax_paid_to_payroll_ratio is reused.

add_feature(
    "estimated_tax_bill / NULLIF(taxable_payroll, 0)",
    "estimated_tax_to_taxable_payroll_ratio"
)

add_feature(
    "estimated_tax_bill / NULLIF(team_payroll, 0)",
    "estimated_tax_to_payroll_ratio"
)

# Rank actual tax payments among teams with observed values.
# Confirmed zero-tax teams remain in the all-league ranking.
add_feature("""
    CASE WHEN tax_paid IS NOT NULL THEN
        RANK() OVER (
            PARTITION BY season
            ORDER BY tax_paid DESC NULLS LAST
        )
    END
""", "tax_paid_league_rank")

# Only positive-tax teams participate in this ranking.
add_feature("""
    CASE WHEN tax_paid > 0 THEN
        RANK() OVER (
            PARTITION BY season, (tax_paid > 0)
            ORDER BY tax_paid DESC NULLS LAST
        )
    END
""", "tax_paid_taxpayer_rank")

add_feature("""
    CASE WHEN tax_paid IS NOT NULL THEN
        tax_paid /
        NULLIF(SUM(tax_paid) OVER (PARTITION BY season), 0)
    END
""", "tax_paid_league_share")

# Estimated-tax rankings remain separate from actual taxes.
add_feature("""
    CASE WHEN estimated_tax_bill IS NOT NULL THEN
        RANK() OVER (
            PARTITION BY season
            ORDER BY estimated_tax_bill DESC NULLS LAST
        )
    END
""", "estimated_tax_bill_rank")

add_feature("""
    CASE WHEN estimated_tax_bill IS NOT NULL THEN
        estimated_tax_bill /
        NULLIF(
            SUM(estimated_tax_bill) OVER (PARTITION BY season),
            0
        )
    END
""", "estimated_tax_bill_observed_share")


# ---------------------------------------------------------
# D2 — TOTAL FINANCIAL COMMITMENT
# ---------------------------------------------------------

add_feature(
    "total_estimated_cost / NULLIF(team_payroll, 0)",
    "total_estimated_cost_to_payroll_ratio"
)

add_feature(
    "total_estimated_cost / NULLIF(taxable_payroll, 0)",
    "total_estimated_cost_to_taxable_payroll_ratio"
)

add_feature("""
    CASE WHEN total_estimated_cost IS NOT NULL THEN
        RANK() OVER (
            PARTITION BY season
            ORDER BY total_estimated_cost DESC NULLS LAST
        )
    END
""", "total_estimated_cost_rank")

add_feature("""
    CASE WHEN total_estimated_cost IS NOT NULL THEN
        PERCENT_RANK() OVER (
            PARTITION BY season,
                         (total_estimated_cost IS NOT NULL)
            ORDER BY total_estimated_cost ASC NULLS LAST
        )
    END
""", "total_estimated_cost_percentile")

add_feature("""
    total_estimated_cost /
    NULLIF(
        MEDIAN(total_estimated_cost) OVER (PARTITION BY season),
        0
    )
""", "total_estimated_cost_to_league_median")

add_feature("""
    total_estimated_cost /
    NULLIF(
        AVG(total_estimated_cost) OVER (PARTITION BY season),
        0
    )
""", "total_estimated_cost_to_league_mean")

# This measures the premium above taxable payroll.
# It should not be labeled an actual-tax payment.
add_feature("""
    (total_estimated_cost - taxable_payroll) /
    NULLIF(taxable_payroll, 0)
""", "estimated_cost_premium_over_taxable_payroll")


# ---------------------------------------------------------
# D3 — THRESHOLD AND REGULATORY EXPOSURE
# ---------------------------------------------------------

# Existing dollar distances are reused.
# Ratio - 1 gives signed proportional distance:
# positive = above threshold; negative = below threshold.

add_feature("""
    taxable_payroll_to_tax_ratio - 1
""", "tax_threshold_distance_pct")

add_feature("""
    first_apron_allocation_ratio - 1
""", "first_apron_distance_pct")

add_feature("""
    second_apron_allocation_ratio - 1
""", "second_apron_distance_pct")

# Regulatory category based on observed accounting position.
# Historical applicability follows available threshold data.
add_feature("""
    CASE
        WHEN taxable_payroll IS NULL
          OR league_luxury_tax_threshold IS NULL
            THEN NULL

        WHEN second_apron_applicable IS TRUE
          AND second_apron_threshold IS NOT NULL
          AND total_apron_allocations >= second_apron_threshold
            THEN 'at_or_above_second_apron'

        WHEN first_apron_threshold IS NOT NULL
          AND total_apron_allocations >= first_apron_threshold
            THEN 'at_or_above_first_apron'

        WHEN taxable_payroll >= league_luxury_tax_threshold
            THEN 'at_or_above_tax_threshold'

        ELSE 'below_tax_threshold'
    END
""", "financial_exposure_category")


# ---------------------------------------------------------
# D4 — LEAGUE-RELATIVE TAX EXPOSURE
# ---------------------------------------------------------

# All-league statistics use observed tax values,
# including confirmed zeros, excluding missing records.
add_feature("""
    CASE WHEN tax_paid IS NOT NULL THEN
        tax_paid -
        AVG(tax_paid) OVER (PARTITION BY season)
    END
""", "tax_paid_vs_league_mean")

add_feature("""
    CASE WHEN tax_paid IS NOT NULL THEN
        tax_paid -
        MEDIAN(tax_paid) OVER (PARTITION BY season)
    END
""", "tax_paid_vs_league_median")

# Taxpayer-only comparisons use strictly positive tax bills.
add_feature("""
    CASE WHEN tax_paid > 0 THEN
        tax_paid -
        AVG(tax_paid) FILTER (
            WHERE tax_paid > 0
        ) OVER (PARTITION BY season)
    END
""", "tax_paid_vs_taxpayer_mean")

add_feature("""
    CASE WHEN tax_paid > 0 THEN
        tax_paid -
        MEDIAN(tax_paid) FILTER (
            WHERE tax_paid > 0
        ) OVER (PARTITION BY season)
    END
""", "tax_paid_vs_taxpayer_median")

add_feature("""
    CASE WHEN tax_paid IS NOT NULL THEN
        PERCENT_RANK() OVER (
            PARTITION BY season, (tax_paid IS NOT NULL)
            ORDER BY tax_paid ASC NULLS LAST
        )
    END
""", "tax_paid_percentile")

add_feature("""
    CASE WHEN tax_paid > 0 THEN
        PERCENT_RANK() OVER (
            PARTITION BY season, (tax_paid > 0)
            ORDER BY tax_paid ASC NULLS LAST
        )
    END
""", "tax_paid_taxpayer_percentile")

# The proportion of franchises paying tax in each season.
# Return NULL when the season lacks complete tax observations.
add_feature("""
    CASE
        WHEN COUNT(tax_paid) OVER (PARTITION BY season)
           = COUNT(*) OVER (PARTITION BY season)
        THEN
            COUNT(*) FILTER (
                WHERE tax_paid > 0
            ) OVER (PARTITION BY season) * 1.0
            /
            NULLIF(COUNT(*) OVER (PARTITION BY season), 0)
    END
""", "taxpayer_share_of_league")

# Relative tax burden uses actual tax / team payroll,
# because actual tax and taxable payroll do not overlap.
add_feature("""
    CASE WHEN tax_paid > 0 THEN
        tax_paid_to_payroll_ratio -
        MEDIAN(tax_paid_to_payroll_ratio) FILTER (
            WHERE tax_paid > 0
        ) OVER (PARTITION BY season)
    END
""", "tax_burden_vs_taxpayer_median")


# ---------------------------------------------------------
# CREATE VIEW
# ---------------------------------------------------------

assert len(d_features) == 26, (
    f"Expected 26 Group D features, found {len(d_features)}"
)

d_sql = ",\n".join(d_features)

sql_a01_v2a_exposure = f"""
CREATE OR REPLACE TEMP VIEW a01_v2a_exposure AS
SELECT
    b.*,
    {d_sql}
FROM a01_v2a_efficiency AS b
"""

con.execute(sql_a01_v2a_exposure).fetchall()

print("Group D view created successfully.")
print("New features:", len(d_features))
print("Provisional engineered-feature total:", 63 + len(d_features))

print("\nNew feature names:")
for item in d_features:
    print(item.split(" AS ")[-1].strip())


Group D view created successfully.
New features: 26
Provisional engineered-feature total: 89

New feature names:
estimated_tax_to_taxable_payroll_ratio
estimated_tax_to_payroll_ratio
tax_paid_league_rank
tax_paid_taxpayer_rank
tax_paid_league_share
estimated_tax_bill_rank
estimated_tax_bill_observed_share
total_estimated_cost_to_payroll_ratio
total_estimated_cost_to_taxable_payroll_ratio
total_estimated_cost_rank
total_estimated_cost_percentile
total_estimated_cost_to_league_median
total_estimated_cost_to_league_mean
estimated_cost_premium_over_taxable_payroll
tax_threshold_distance_pct
first_apron_distance_pct
second_apron_distance_pct
financial_exposure_category
tax_paid_vs_league_mean
tax_paid_vs_league_median
tax_paid_vs_taxpayer_mean
tax_paid_vs_taxpayer_median
tax_paid_percentile
tax_paid_taxpayer_percentile
taxpayer_share_of_league
tax_burden_vs_taxpayer_median


In [44]:

# A01 v2A — Group D: Consolidated Audit
# Structural integrity, feature coverage, formula checks,
# and source-accounting consistency.

# ---------------------------------------------------------
# 1. STRUCTURAL INTEGRITY
# ---------------------------------------------------------

structure = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys
    FROM a01_v2a_exposure
""").df()

print("1. STRUCTURAL INTEGRITY")
display(structure)


# ---------------------------------------------------------
# 2. COVERAGE OF ALL NEW FEATURES
# ---------------------------------------------------------

feature_names = [
    item.split(" AS ")[-1].strip()
    for item in d_features
]

coverage_sql = ",\n".join(
    f'COUNT("{name}") AS "{name}"'
    for name in feature_names
)

coverage = con.sql(f"""
    SELECT {coverage_sql}
    FROM a01_v2a_exposure
""").df()

print("\n2. FEATURE COVERAGE")
display(
    coverage.T.rename(columns={0: "non_null_rows"})
)


# ---------------------------------------------------------
# 3. TOTAL COST ACCOUNTING CHECK
# ---------------------------------------------------------

cost_audit = con.sql("""
    SELECT
        COUNT(total_estimated_cost) AS observed_cost_rows,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND ABS(
                  total_estimated_cost -
                  (
                      taxable_payroll +
                      COALESCE(estimated_tax_bill, 0)
                  )
              ) > 1
        ) AS cost_formula_mismatches,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND estimated_tax_bill IS NULL
        ) AS cost_rows_with_missing_bill

    FROM a01_v2a_exposure
""").df()

print("\n3. TOTAL COST ACCOUNTING")
display(cost_audit)


# ---------------------------------------------------------
# 4. ARITHMETIC AND LOGICAL CHECKS
# ---------------------------------------------------------

checks = con.sql("""
    SELECT

        COUNT(*) FILTER (
            WHERE estimated_tax_bill IS NOT NULL
              AND taxable_payroll > 0
              AND ABS(
                  estimated_tax_to_taxable_payroll_ratio
                  - estimated_tax_bill / taxable_payroll
              ) > 0.000001
        ) AS estimated_tax_ratio_errors,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NOT NULL
              AND taxable_payroll > 0
              AND ABS(
                  total_estimated_cost_to_taxable_payroll_ratio
                  - total_estimated_cost / taxable_payroll
              ) > 0.000001
        ) AS total_cost_ratio_errors,

        COUNT(*) FILTER (
            WHERE tax_threshold_distance_pct IS NOT NULL
              AND ABS(
                  tax_threshold_distance_pct -
                  (taxable_payroll_to_tax_ratio - 1)
              ) > 0.000001
        ) AS tax_distance_pct_errors,

        COUNT(*) FILTER (
            WHERE first_apron_distance_pct IS NOT NULL
              AND ABS(
                  first_apron_distance_pct -
                  (first_apron_allocation_ratio - 1)
              ) > 0.000001
        ) AS first_apron_pct_errors,

        COUNT(*) FILTER (
            WHERE second_apron_distance_pct IS NOT NULL
              AND ABS(
                  second_apron_distance_pct -
                  (second_apron_allocation_ratio - 1)
              ) > 0.000001
        ) AS second_apron_pct_errors,

        COUNT(*) FILTER (
            WHERE tax_paid IS NULL
              AND tax_paid_league_rank IS NOT NULL
        ) AS missing_actual_tax_rank_errors,

        COUNT(*) FILTER (
            WHERE (tax_paid IS NULL OR tax_paid <= 0)
              AND tax_paid_taxpayer_rank IS NOT NULL
        ) AS taxpayer_rank_errors,

        COUNT(*) FILTER (
            WHERE estimated_tax_bill IS NULL
              AND estimated_tax_bill_rank IS NOT NULL
        ) AS missing_estimated_tax_rank_errors,

        COUNT(*) FILTER (
            WHERE total_estimated_cost IS NULL
              AND total_estimated_cost_rank IS NOT NULL
        ) AS missing_total_cost_rank_errors

    FROM a01_v2a_exposure
""").df()

print("\n4. ARITHMETIC AND LOGICAL CHECKS")
display(checks.T.rename(columns={0: "error_count"}))


# ---------------------------------------------------------
# 5. COVERAGE BY FINANCIAL ERA
# ---------------------------------------------------------

era_coverage = con.sql("""
    SELECT
        season,
        COUNT(*) AS teams,
        COUNT(tax_paid_league_rank) AS actual_tax_rank_n,
        COUNT(estimated_tax_bill_rank) AS estimated_tax_rank_n,
        COUNT(total_estimated_cost_rank) AS total_cost_rank_n,
        COUNT(financial_exposure_category) AS category_n,
        MAX(taxpayer_share_of_league) AS taxpayer_share

    FROM a01_v2a_exposure
    GROUP BY season
    ORDER BY season
""").df()

print("\n5. SEASON-LEVEL COVERAGE")
display(era_coverage)

print("\nGroup D audit completed.")


1. STRUCTURAL INTEGRITY


,total_rows,unique_keys
0,776,776



2. FEATURE COVERAGE


,non_null_rows
estimated_tax_to_taxable_payroll_ratio,94
estimated_tax_to_payroll_ratio,94
tax_paid_league_rank,268
tax_paid_taxpayer_rank,72
tax_paid_league_share,268
estimated_tax_bill_rank,94
estimated_tax_bill_observed_share,94
total_estimated_cost_to_payroll_ratio,450
total_estimated_cost_to_taxable_payroll_ratio,450
total_estimated_cost_rank,450



3. TOTAL COST ACCOUNTING


,observed_cost_rows,cost_formula_mismatches,cost_rows_with_missing_bill
0,450,0,356



4. ARITHMETIC AND LOGICAL CHECKS


,error_count
estimated_tax_ratio_errors,0
total_cost_ratio_errors,0
tax_distance_pct_errors,0
first_apron_pct_errors,0
second_apron_pct_errors,0
missing_actual_tax_rank_errors,0
taxpayer_rank_errors,0
missing_estimated_tax_rank_errors,0
missing_total_cost_rank_errors,0



5. SEASON-LEVEL COVERAGE


,season,teams,actual_tax_rank_n,estimated_tax_rank_n,total_cost_rank_n,category_n,taxpayer_share
0,2000-2001,29,0,0,0,0,NaN
1,2001-2002,29,0,0,0,0,NaN
2,2002-2003,29,29,0,0,0,0.551724
3,2003-2004,29,29,0,0,0,0.379310
4,2004-2005,30,30,0,0,0,0.033333
5,2005-2006,30,30,0,0,0,0.200000
6,2006-2007,30,30,0,0,0,0.166667
7,2007-2008,30,30,0,0,0,0.266667
8,2008-2009,30,30,0,0,0,0.233333
9,2009-2010,30,30,0,0,0,0.366667



Group D audit completed.


In [45]:

# A01 v2B — Year-over-Year Features
# Input:  a01_v2a_exposure
# Output: a01_v2b_yoy
#
# 14 features:
# 3 spending growth, 4 boundary growth,
# 4 relative financial-position changes,
# 2 competitive changes, 1 efficiency change.

sql_a01_v2b_yoy = """
CREATE OR REPLACE TEMP VIEW a01_v2b_yoy AS

WITH season_boundaries AS (
    -- League boundaries belong to seasons, not individual teams.
    -- DISTINCT produces one record per season.
    SELECT DISTINCT
        season,
        CAST(SPLIT_PART(season, '-', 1) AS INTEGER)
            AS season_year,
        salary_cap,
        league_luxury_tax_threshold,
        league_first_apron,
        league_second_apron
    FROM a01_v2a_exposure
),

boundary_lags AS (
    SELECT
        *,
        LAG(season_year) OVER w AS prev_season_year,
        LAG(salary_cap) OVER w AS prev_salary_cap,
        LAG(league_luxury_tax_threshold) OVER w
            AS prev_tax_threshold,
        LAG(league_first_apron) OVER w
            AS prev_first_apron,
        LAG(league_second_apron) OVER w
            AS prev_second_apron
    FROM season_boundaries
    WINDOW w AS (ORDER BY season_year)
),

boundary_growth AS (
    SELECT
        season,

        -- Only compare consecutive seasons.
        CASE WHEN prev_season_year = season_year - 1
        THEN salary_cap / NULLIF(prev_salary_cap, 0) - 1
        END AS salary_cap_yoy_growth,

        CASE WHEN prev_season_year = season_year - 1
        THEN league_luxury_tax_threshold
             / NULLIF(prev_tax_threshold, 0) - 1
        END AS tax_threshold_yoy_growth,

        CASE WHEN prev_season_year = season_year - 1
        THEN league_first_apron
             / NULLIF(prev_first_apron, 0) - 1
        END AS first_apron_threshold_yoy_growth,

        CASE WHEN prev_season_year = season_year - 1
        THEN league_second_apron
             / NULLIF(prev_second_apron, 0) - 1
        END AS second_apron_threshold_yoy_growth

    FROM boundary_lags
),

franchise_lags AS (
    SELECT
        b.*,

        CAST(SPLIT_PART(season, '-', 1) AS INTEGER)
            AS current_year,

        -- PARTITION BY keeps each franchise's history separate.
        -- ORDER BY places its seasons chronologically.
        LAG(
            CAST(SPLIT_PART(season, '-', 1) AS INTEGER)
        ) OVER w AS prev_year,

        LAG(team_payroll) OVER w AS prev_team_payroll,
        LAG(taxable_payroll) OVER w AS prev_taxable_payroll,
        LAG(total_apron_allocations) OVER w
            AS prev_apron_allocations,

        LAG(payroll_to_cap_ratio) OVER w
            AS prev_payroll_cap_ratio,
        LAG(taxable_payroll_to_tax_ratio) OVER w
            AS prev_taxable_tax_ratio,
        LAG(first_apron_allocation_ratio) OVER w
            AS prev_first_apron_ratio,
        LAG(second_apron_allocation_ratio) OVER w
            AS prev_second_apron_ratio,

        LAG(win_pct) OVER w AS prev_win_pct,
        LAG(payroll_rank) OVER w AS prev_payroll_rank,
        LAG(payroll_normalized_efficiency) OVER w
            AS prev_payroll_efficiency

    FROM a01_v2a_exposure AS b

    WINDOW w AS (
        PARTITION BY franchise_id
        ORDER BY CAST(SPLIT_PART(season, '-', 1) AS INTEGER)
    )
)

SELECT
    -- Preserve all original A01 columns.
    -- EXCLUDE removes temporary lag helper columns.
    f.* EXCLUDE (
        current_year, prev_year,
        prev_team_payroll, prev_taxable_payroll,
        prev_apron_allocations,
        prev_payroll_cap_ratio, prev_taxable_tax_ratio,
        prev_first_apron_ratio, prev_second_apron_ratio,
        prev_win_pct, prev_payroll_rank,
        prev_payroll_efficiency
    ),

    -- GROUP A: Nominal financial growth (3)
    CASE WHEN prev_year = current_year - 1
    THEN team_payroll / NULLIF(prev_team_payroll, 0) - 1
    END AS team_payroll_yoy_growth,

    CASE WHEN prev_year = current_year - 1
    THEN taxable_payroll / NULLIF(prev_taxable_payroll, 0) - 1
    END AS taxable_payroll_yoy_growth,

    CASE WHEN prev_year = current_year - 1
    THEN total_apron_allocations
         / NULLIF(prev_apron_allocations, 0) - 1
    END AS apron_allocations_yoy_growth,

    -- GROUP B: League boundary growth (4)
    g.salary_cap_yoy_growth,
    g.tax_threshold_yoy_growth,
    g.first_apron_threshold_yoy_growth,
    g.second_apron_threshold_yoy_growth,

    -- GROUP C: Changes in relative financial position (4)
    -- These are ratio-point changes, not percentage growth.
    -- They also equal changes in normalized threshold distance.
    CASE WHEN prev_year = current_year - 1
    THEN payroll_to_cap_ratio - prev_payroll_cap_ratio
    END AS payroll_to_cap_yoy_change,

    CASE WHEN prev_year = current_year - 1
    THEN taxable_payroll_to_tax_ratio - prev_taxable_tax_ratio
    END AS taxable_to_tax_yoy_change,

    CASE WHEN prev_year = current_year - 1
    THEN first_apron_allocation_ratio - prev_first_apron_ratio
    END AS first_apron_ratio_yoy_change,

    CASE WHEN prev_year = current_year - 1
    THEN second_apron_allocation_ratio - prev_second_apron_ratio
    END AS second_apron_ratio_yoy_change,

    -- GROUP D: Competitive changes (2)
    CASE WHEN prev_year = current_year - 1
    THEN win_pct - prev_win_pct
    END AS win_pct_yoy_change,

    -- Positive means an improvement in payroll rank.
    CASE WHEN prev_year = current_year - 1
    THEN prev_payroll_rank - payroll_rank
    END AS payroll_rank_yoy_improvement,

    -- GROUP E: Spending-efficiency change (1)
    CASE WHEN prev_year = current_year - 1
    THEN payroll_normalized_efficiency
         - prev_payroll_efficiency
    END AS payroll_efficiency_yoy_change

FROM franchise_lags AS f
LEFT JOIN boundary_growth AS g
    ON f.season = g.season
"""

con.execute(sql_a01_v2b_yoy).fetchall()

print("Created a01_v2b_yoy with 14 YoY features.")


Created a01_v2b_yoy with 14 YoY features.


In [46]:

# A01 v2B — Consolidated structural and coverage audit

v2b_features = [
    "team_payroll_yoy_growth",
    "taxable_payroll_yoy_growth",
    "apron_allocations_yoy_growth",
    "salary_cap_yoy_growth",
    "tax_threshold_yoy_growth",
    "first_apron_threshold_yoy_growth",
    "second_apron_threshold_yoy_growth",
    "payroll_to_cap_yoy_change",
    "taxable_to_tax_yoy_change",
    "first_apron_ratio_yoy_change",
    "second_apron_ratio_yoy_change",
    "win_pct_yoy_change",
    "payroll_rank_yoy_improvement",
    "payroll_efficiency_yoy_change"
]

# Confirm we implemented exactly the selected 14 features.
assert len(v2b_features) == 14

structure = con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys
    FROM a01_v2b_yoy
""").df()

display(structure)

assert structure["rows"].iloc[0] == 776
assert structure["unique_keys"].iloc[0] == 776

# Check that all previous columns are preserved and exactly
# 14 additional columns were introduced.
old_columns = con.sql(
    "DESCRIBE SELECT * FROM a01_v2a_exposure"
).df()["column_name"].tolist()

new_columns = con.sql(
    "DESCRIBE SELECT * FROM a01_v2b_yoy"
).df()["column_name"].tolist()

assert all(c in new_columns for c in old_columns)
assert set(new_columns) - set(old_columns) == set(v2b_features)

# Count non-null observations for every new feature.
coverage_sql = ", ".join(
    f'COUNT("{name}") AS "{name}"'
    for name in v2b_features
)

coverage = con.sql(f"""
    SELECT {coverage_sql}
    FROM a01_v2b_yoy
""").df().T

coverage.columns = ["non_null_rows"]

display(coverage)

print("v2B structural and coverage checks passed.")


,rows,unique_keys
0,776,776


,non_null_rows
team_payroll_yoy_growth,745
taxable_payroll_yoy_growth,420
apron_allocations_yoy_growth,420
salary_cap_yoy_growth,747
tax_threshold_yoy_growth,629
first_apron_threshold_yoy_growth,420
second_apron_threshold_yoy_growth,60
payroll_to_cap_yoy_change,745
taxable_to_tax_yoy_change,420
first_apron_ratio_yoy_change,420


v2B structural and coverage checks passed.


In [47]:

# A01 v2B — Final targeted arithmetic audit
# Independently recompute selected YoY features from v2A inputs.
# IS DISTINCT FROM treats matching NULL values as equal,
# while flagging a NULL-versus-non-NULL mismatch.

audit = con.sql("""
WITH prior AS (
    SELECT
        *,
        CAST(SPLIT_PART(season, '-', 1) AS INTEGER) AS yr,

        LAG(CAST(SPLIT_PART(season, '-', 1) AS INTEGER))
            OVER w AS prev_yr,

        LAG(team_payroll) OVER w AS prev_payroll,
        LAG(win_pct) OVER w AS prev_win_pct,
        LAG(payroll_rank) OVER w AS prev_rank,
        LAG(tax_threshold_distance_pct) OVER w AS prev_tax_dist,
        LAG(first_apron_distance_pct) OVER w AS prev_first_dist,
        LAG(second_apron_distance_pct) OVER w AS prev_second_dist

    FROM a01_v2a_exposure

    WINDOW w AS (
        PARTITION BY franchise_id
        ORDER BY CAST(SPLIT_PART(season, '-', 1) AS INTEGER)
    )
),

expected AS (
    SELECT
        franchise_id,
        season,

        CASE WHEN prev_yr = yr - 1
        THEN team_payroll / NULLIF(prev_payroll, 0) - 1
        END AS expected_payroll_growth,

        CASE WHEN prev_yr = yr - 1
        THEN win_pct - prev_win_pct
        END AS expected_win_change,

        CASE WHEN prev_yr = yr - 1
        THEN prev_rank - payroll_rank
        END AS expected_rank_change,

        CASE WHEN prev_yr = yr - 1
        THEN tax_threshold_distance_pct - prev_tax_dist
        END AS expected_tax_change,

        CASE WHEN prev_yr = yr - 1
        THEN first_apron_distance_pct - prev_first_dist
        END AS expected_first_change,

        CASE WHEN prev_yr = yr - 1
        THEN second_apron_distance_pct - prev_second_dist
        END AS expected_second_change

    FROM prior
),

checks AS (
    SELECT
        v.franchise_id,
        v.season,

        -- ROUND avoids false mismatches from tiny
        -- floating-point differences.
        ROUND(v.team_payroll_yoy_growth, 9)
            IS DISTINCT FROM ROUND(e.expected_payroll_growth, 9)
            AS payroll_mismatch,

        ROUND(v.win_pct_yoy_change, 9)
            IS DISTINCT FROM ROUND(e.expected_win_change, 9)
            AS win_mismatch,

        v.payroll_rank_yoy_improvement
            IS DISTINCT FROM e.expected_rank_change
            AS rank_mismatch,

        ROUND(v.taxable_to_tax_yoy_change, 9)
            IS DISTINCT FROM ROUND(e.expected_tax_change, 9)
            AS tax_mismatch,

        ROUND(v.first_apron_ratio_yoy_change, 9)
            IS DISTINCT FROM ROUND(e.expected_first_change, 9)
            AS first_apron_mismatch,

        ROUND(v.second_apron_ratio_yoy_change, 9)
            IS DISTINCT FROM ROUND(e.expected_second_change, 9)
            AS second_apron_mismatch

    FROM a01_v2b_yoy v
    JOIN expected e
      ON v.franchise_id = e.franchise_id
     AND v.season = e.season
)

SELECT
    COUNT(*) AS rows_checked,
    SUM(CAST(payroll_mismatch AS INTEGER)) AS payroll_errors,
    SUM(CAST(win_mismatch AS INTEGER)) AS win_errors,
    SUM(CAST(rank_mismatch AS INTEGER)) AS rank_errors,
    SUM(CAST(tax_mismatch AS INTEGER)) AS tax_errors,
    SUM(CAST(first_apron_mismatch AS INTEGER)) AS first_apron_errors,
    SUM(CAST(second_apron_mismatch AS INTEGER)) AS second_apron_errors
FROM checks
""").df()

display(audit)

# All six independent checks must have zero mismatches.
assert audit.iloc[0]["rows_checked"] == 776
assert audit.iloc[0, 1:].eq(0).all()

print("PASS: v2B targeted arithmetic checks.")


,rows_checked,payroll_errors,win_errors,rank_errors,tax_errors,first_apron_errors,second_apron_errors
0,776,0.0,0.0,0.0,0.0,0.0,0.0


PASS: v2B targeted arithmetic checks.


In [48]:

# ============================================================
# A01 v2C — Multiyear Trends & Persistence (CORRECTED)
#
# Input:  a01_v2b_yoy
# Output: a01_v2c_rolling
#
# Uses the EXISTING season_year column from v2B.
# Adds 22 features using 3- and 6-season windows.
# ============================================================

# A. Financial and competitive measures to average.
average_measures = {
    "payroll_to_cap_ratio": "avg_payroll_to_cap",
    "taxable_payroll_to_tax_ratio": "avg_taxable_to_tax",
    "first_apron_allocation_ratio": "avg_first_apron_ratio",
    "second_apron_allocation_ratio": "avg_second_apron_ratio",
    "win_pct": "avg_win_pct",
    "payroll_normalized_efficiency": "avg_payroll_efficiency"
}

# B. Playoff milestones to count.
playoff_measures = {
    "made_playoffs": "playoff_appearances",
    "reached_second_round": "second_round_appearances",
    "reached_conference_finals": "conference_finals_appearances"
}

# C. Financial threshold positions to count.
financial_persistence = {
    "taxable_payroll_to_tax_ratio": "tax_threshold_seasons",
    "first_apron_allocation_ratio": "first_apron_seasons",
    "second_apron_allocation_ratio": "second_apron_seasons"
}

# Collect the SQL expressions for all 22 features.
v2c_features = []

for n in (3, 6):

    # w3 and w6 are defined in the SQL WINDOW clause below.
    w = f"w{n}"

    # A valid rolling window requires n consecutive seasons.
    complete_window = (
        f"COUNT(*) OVER {w} = {n} "
        f"AND MAX(season_year) OVER {w} "
        f"- MIN(season_year) OVER {w} = {n - 1}"
    )

    # Rolling averages.
    for source, prefix in average_measures.items():

        # Insufficient history for a six-season second-apron measure.
        if n == 6 and source == "second_apron_allocation_ratio":
            continue

        name = f"{prefix}_{n}yr"

        expression = f"""
        CASE
            WHEN {complete_window}
             AND COUNT({source}) OVER {w} = {n}
            THEN AVG({source}) OVER {w}
        END AS {name}
        """

        v2c_features.append((name, expression))

    # Rolling playoff counts.
    for source, prefix in playoff_measures.items():

        name = f"{prefix}_{n}yr"

        expression = f"""
        CASE
            WHEN {complete_window}
             AND COUNT({source}) OVER {w} = {n}
            THEN SUM(
                CASE WHEN {source} THEN 1 ELSE 0 END
            ) OVER {w}
        END AS {name}
        """

        v2c_features.append((name, expression))

    # Rolling financial persistence counts.
    for source, prefix in financial_persistence.items():

        if n == 6 and source == "second_apron_allocation_ratio":
            continue

        name = f"{prefix}_{n}yr"

        expression = f"""
        CASE
            WHEN {complete_window}
             AND COUNT({source}) OVER {w} = {n}
            THEN SUM(
                CASE WHEN {source} >= 1 THEN 1 ELSE 0 END
            ) OVER {w}
        END AS {name}
        """

        v2c_features.append((name, expression))


# Confirm we generated exactly 22 features.
assert len(v2c_features) == 22

# Combine the expressions into the SELECT statement.
feature_sql = ",\n".join(
    expression for name, expression in v2c_features
)

# IMPORTANT CORRECTION:
# v2B already contains season_year.
# We use that existing column rather than create a duplicate.
# SELECT b.* preserves every existing v2B column.

sql_a01_v2c = f"""
CREATE OR REPLACE TEMP VIEW a01_v2c_rolling AS

SELECT
    b.*,

    {feature_sql}

FROM a01_v2b_yoy AS b

WINDOW
    w3 AS (
        PARTITION BY franchise_id
        ORDER BY season_year
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ),

    w6 AS (
        PARTITION BY franchise_id
        ORDER BY season_year
        ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
    )
"""

# Replace the existing temporary view with the corrected one.
con.execute(sql_a01_v2c).fetchall()

print("Created corrected a01_v2c_rolling")
print("New features:", len(v2c_features))
print("Provisional engineered-feature total:", 103 + len(v2c_features))

# Quick schema confirmation.
old_columns = con.sql("""
    DESCRIBE SELECT * FROM a01_v2b_yoy
""").df()["column_name"].tolist()

new_columns = con.sql("""
    DESCRIBE SELECT * FROM a01_v2c_rolling
""").df()["column_name"].tolist()

feature_names = [name for name, expression in v2c_features]

assert set(old_columns).issubset(set(new_columns))
assert set(new_columns) - set(old_columns) == set(feature_names)
assert len(new_columns) - len(old_columns) == 22

print("PASS: All original v2B columns preserved.")
print("PASS: Exactly 22 new v2C columns.")


Created corrected a01_v2c_rolling
New features: 22
Provisional engineered-feature total: 125
PASS: All original v2B columns preserved.
PASS: Exactly 22 new v2C columns.


In [49]:

# ============================================================
# A01 v2C — Structural & Coverage Audit (FINAL)
# ============================================================

# 1. Verify row count and unique franchise-season keys.
structure = con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys
    FROM a01_v2c_rolling
""").df()

display(structure)

assert structure["rows"].iloc[0] == 776
assert structure["unique_keys"].iloc[0] == 776

print("PASS: Row count and unique keys.")


# 2. Verify that all v2B columns were preserved.
old_columns = con.sql("""
    DESCRIBE SELECT * FROM a01_v2b_yoy
""").df()["column_name"].tolist()

new_columns = con.sql("""
    DESCRIBE SELECT * FROM a01_v2c_rolling
""").df()["column_name"].tolist()

feature_names = [name for name, expression in v2c_features]

assert set(old_columns).issubset(set(new_columns))
assert set(new_columns) - set(old_columns) == set(feature_names)
assert len(new_columns) - len(old_columns) == 22

print("PASS: Original columns preserved.")
print("PASS: Exactly 22 new v2C features.")


# 3. Calculate coverage for all 22 features.
coverage_sql = ",\n".join(
    f'COUNT("{name}") AS "{name}"'
    for name in feature_names
)

coverage = con.sql(f"""
    SELECT {coverage_sql}
    FROM a01_v2c_rolling
""").df().T

coverage.columns = ["non_null_rows"]

display(coverage)

print("PASS: Feature coverage calculated.")


# 4. Verify all rolling counts are within valid ranges.
count_features = [
    name for name in feature_names
    if "appearances" in name or "_seasons_" in name
]

invalid_count_checks = " + ".join(
    f"""COUNT(*) FILTER (
        WHERE "{name}" < 0
           OR "{name}" > {3 if name.endswith("3yr") else 6}
    )"""
    for name in count_features
)

# Use .df() rather than .fetchone() for compatibility.
validation = con.sql(f"""
    SELECT
        {invalid_count_checks} AS invalid_values
    FROM a01_v2c_rolling
""").df()

display(validation)

assert validation["invalid_values"].iloc[0] == 0

print("PASS: All rolling counts within valid ranges.")
print("PASS: A01 v2C structural and coverage audit complete.")


,rows,unique_keys
0,776,776


PASS: Row count and unique keys.
PASS: Original columns preserved.
PASS: Exactly 22 new v2C features.


,non_null_rows
avg_payroll_to_cap_3yr,714
avg_taxable_to_tax_3yr,390
avg_first_apron_ratio_3yr,390
avg_second_apron_ratio_3yr,30
avg_win_pct_3yr,714
avg_payroll_efficiency_3yr,714
playoff_appearances_3yr,714
second_round_appearances_3yr,714
conference_finals_appearances_3yr,714
tax_threshold_seasons_3yr,390


PASS: Feature coverage calculated.


,invalid_values
0,0


PASS: All rolling counts within valid ranges.
PASS: A01 v2C structural and coverage audit complete.


In [50]:

# ============================================================
# A01 v2C — Final Arithmetic Validation
#
# Independently recomputes selected rolling features
# using pandas, rather than the SQL window expressions.
#
# Checks:
# - 3-year payroll-to-cap average
# - 6-year winning-percentage average
# - 3-year playoff appearance count
# - 6-year conference finals appearance count
# - 3-year tax-threshold persistence count
# - 6-year first-apron persistence count
# ============================================================

import pandas as pd
import numpy as np

# Retrieve source data from v2B.
# This deliberately does not use the v2C calculations.
source = con.sql("""
    SELECT
        franchise_id,
        season,
        season_year,
        payroll_to_cap_ratio,
        win_pct,
        made_playoffs,
        reached_conference_finals,
        taxable_payroll_to_tax_ratio,
        first_apron_allocation_ratio
    FROM a01_v2b_yoy
    ORDER BY franchise_id, season_year
""").df()

# Retrieve only the v2C outputs we're testing.
actual = con.sql("""
    SELECT
        franchise_id,
        season,
        avg_payroll_to_cap_3yr,
        avg_win_pct_6yr,
        playoff_appearances_3yr,
        conference_finals_appearances_6yr,
        tax_threshold_seasons_3yr,
        first_apron_seasons_6yr
    FROM a01_v2c_rolling
""").df()

# Recalculate features using pandas rolling windows.
# min_periods=n requires n non-null observations.
# We also verify that the seasons are consecutive.
checks = [
    ("payroll_to_cap_ratio", "avg_payroll_to_cap_3yr", 3, "mean"),
    ("win_pct", "avg_win_pct_6yr", 6, "mean"),
    ("made_playoffs", "playoff_appearances_3yr", 3, "count_true"),
    ("reached_conference_finals", "conference_finals_appearances_6yr", 6, "count_true"),
    ("taxable_payroll_to_tax_ratio", "tax_threshold_seasons_3yr", 3, "count_threshold"),
    ("first_apron_allocation_ratio", "first_apron_seasons_6yr", 6, "count_threshold")
]

expected_rows = []

# Each franchise is processed separately.
for franchise_id, group in source.groupby("franchise_id"):

    group = group.sort_values("season_year").reset_index(drop=True)

    for i in range(len(group)):

        result = {
            "franchise_id": franchise_id,
            "season": group.loc[i, "season"]
        }

        for source_col, output_col, n, operation in checks:

            # Default to missing until the window is verified.
            value = np.nan

            if i >= n - 1:

                window = group.iloc[i - n + 1:i + 1]

                # Require n consecutive years.
                years = window["season_year"].to_numpy()

                consecutive = np.all(np.diff(years) == 1)

                observations = window[source_col]

                # All observations must be present.
                complete = observations.notna().all()

                if consecutive and complete:

                    if operation == "mean":
                        value = observations.mean()

                    elif operation == "count_true":
                        value = int(observations.astype(bool).sum())

                    elif operation == "count_threshold":
                        value = int((observations >= 1).sum())

            result[output_col] = value

        expected_rows.append(result)

expected = pd.DataFrame(expected_rows)

# Join the independent results to the v2C outputs.
comparison = actual.merge(
    expected,
    on=["franchise_id", "season"],
    suffixes=("_actual", "_expected"),
    validate="one_to_one"
)

assert len(comparison) == 776

# Compare every independently calculated value.
# np.isclose handles minor floating-point differences.
# equal_nan=True treats two missing values as matching.
results = []

for _, feature, _, _ in checks:

    a = pd.to_numeric(
        comparison[f"{feature}_actual"], errors="coerce"
    ).to_numpy(dtype=float)

    e = pd.to_numeric(
        comparison[f"{feature}_expected"], errors="coerce"
    ).to_numpy(dtype=float)

    mismatches = int(
        (~np.isclose(a, e, rtol=1e-9, atol=1e-9, equal_nan=True)).sum()
    )

    results.append({
        "feature": feature,
        "mismatches": mismatches
    })

audit_results = pd.DataFrame(results)

display(audit_results)

assert audit_results["mismatches"].eq(0).all()

print("PASS: All six independent rolling calculations match.")
print("PASS: A01 v2C arithmetic validation complete.")


,feature,mismatches
0,avg_payroll_to_cap_3yr,0
1,avg_win_pct_6yr,0
2,playoff_appearances_3yr,0
3,conference_finals_appearances_6yr,0
4,tax_threshold_seasons_3yr,0
5,first_apron_seasons_6yr,0


PASS: All six independent rolling calculations match.
PASS: A01 v2C arithmetic validation complete.


In [51]:
# Inspect existing A01 v2C columns that may support state transitions.
# This does not modify any data or create new features.

columns = con.sql("""
    DESCRIBE SELECT *
    FROM a01_v2c_rolling
""").df()

# Show columns related to financial thresholds and postseason outcomes.
keywords = [
    "tax", "apron", "playoff",
    "second_round", "conference_finals",
    "season_year", "franchise_id"
]

mask = columns["column_name"].str.contains(
    "|".join(keywords),
    case=False,
    regex=True
)

display(columns.loc[mask, ["column_name", "column_type"]])

,column_name,column_type
10,franchise_id,BIGINT
36,playoff_series_won,BIGINT
42,made_playoffs,BOOLEAN
43,s18_tax_threshold,DOUBLE
44,taxable_payroll,DOUBLE
...,...,...
193,playoff_appearances_6yr,HUGEINT
194,second_round_appearances_6yr,HUGEINT
195,conference_finals_appearances_6yr,HUGEINT
196,tax_threshold_seasons_6yr,HUGEINT


In [52]:
# A01 v3 — Inspect the exact source columns needed for transitions.
# Read-only: no tables, views, or features are modified.

source_columns = [
    "franchise_id",
    "season",
    "season_year",
    "taxable_payroll_to_tax_ratio",
    "first_apron_allocation_ratio",
    "second_apron_allocation_ratio",
    "made_playoffs",
    "reached_second_round",
    "reached_conference_finals"
]

# Retrieve the existing schema from the v2C view.
schema = con.sql("""
    DESCRIBE SELECT *
    FROM a01_v2c_rolling
""").df()

# Check which expected columns actually exist.
found = schema[
    schema["column_name"].isin(source_columns)
][["column_name", "column_type"]]

missing = [
    name for name in source_columns
    if name not in found["column_name"].tolist()
]

print("A01 v3 — Required source columns")
display(found)

print("\nMissing expected columns:", missing)

# Confirm whether all nine expected source columns are available.
if not missing:
    print("PASS: All expected v3 source columns exist.")
else:
    print("REVIEW: Some source column names need to be reconciled.")

A01 v3 — Required source columns


,column_name,column_type
9,season,VARCHAR
10,franchise_id,BIGINT
42,made_playoffs,BOOLEAN
53,season_year,BIGINT
75,taxable_payroll_to_tax_ratio,DOUBLE
76,first_apron_allocation_ratio,DOUBLE
77,second_apron_allocation_ratio,DOUBLE
85,reached_second_round,BOOLEAN
86,reached_conference_finals,BOOLEAN



Missing expected columns: []
PASS: All expected v3 source columns exist.


In [53]:
# ============================================================
# A01 v3 — Franchise-season state transitions
# ============================================================

# Six financial and six competitive transition features.
# Each pair identifies entry into, or departure from, a state.

financial_states = {
    "tax_threshold": "taxable_payroll_to_tax_ratio >= 1",
    "first_apron": "first_apron_allocation_ratio >= 1",
    "second_apron": "second_apron_allocation_ratio >= 1",
}

competitive_states = {
    "playoffs": "made_playoffs",
    "second_round": "reached_second_round",
    "conference_finals": "reached_conference_finals",
}

# Define the names for each transition pair.
transition_names = {
    "tax_threshold": ("entered_tax_threshold", "exited_tax_threshold"),
    "first_apron": ("entered_first_apron", "exited_first_apron"),
    "second_apron": ("entered_second_apron", "exited_second_apron"),
    "playoffs": ("newly_qualified_playoffs", "lost_playoff_qualification"),
    "second_round": ("newly_reached_second_round", "failed_to_repeat_second_round"),
    "conference_finals": (
        "newly_reached_conference_finals",
        "failed_to_repeat_conference_finals",
    ),
}

all_states = {**financial_states, **competitive_states}

# Construct SQL expressions for the six current-season states.
# NULL input remains NULL, rather than being classified as False.
state_sql = []

for name, expression in all_states.items():
    state_sql.append(
        f"CASE WHEN ({expression.split(' >= ')[0]}) IS NULL "
        f"THEN NULL ELSE ({expression}) END AS state_{name}"
        if name in financial_states
        else f"{expression} AS state_{name}"
    )

# LAG() retrieves the preceding row within each franchise.
# The season-year check prevents nonconsecutive seasons from
# being interpreted as adjacent observations.
previous_sql = [
    "LAG(season_year) OVER w AS previous_season_year"
]

for name in all_states:
    previous_sql.append(
        f"LAG(state_{name}) OVER w AS previous_{name}"
    )

# Build two Boolean indicators for each state.
transition_sql = []

for name, (entry_name, exit_name) in transition_names.items():

    current = f"state_{name}"
    previous = f"previous_{name}"

    valid = (
        f"previous_season_year = season_year - 1 "
        f"AND {current} IS NOT NULL "
        f"AND {previous} IS NOT NULL"
    )

    transition_sql.extend([
        f"""
        CASE
            WHEN {valid}
            THEN (NOT {previous} AND {current})
            ELSE NULL
        END AS {entry_name}
        """,
        f"""
        CASE
            WHEN {valid}
            THEN ({previous} AND NOT {current})
            ELSE NULL
        END AS {exit_name}
        """
    ])

# Assemble the view using CTEs:
# 1. states: classify each franchise-season.
# 2. previous: obtain prior-season states.
# 3. final SELECT: append 12 transition indicators.

build_sql = f"""
CREATE OR REPLACE TEMP VIEW a01_v3_transitions AS

WITH states AS (
    SELECT
        *,
        {", ".join(state_sql)}
    FROM a01_v2c_rolling
),

previous AS (
    SELECT
        *,
        {", ".join(previous_sql)}
    FROM states
    WINDOW w AS (
        PARTITION BY franchise_id
        ORDER BY season_year
    )
)

SELECT
    b.* EXCLUDE (
        {", ".join(["state_" + n for n in all_states])},
        {", ".join(["previous_" + n for n in all_states])},
        previous_season_year
    ),
    {", ".join(transition_sql)}
FROM previous AS b
"""

con.sql(build_sql)

print("PASS: a01_v3_transitions created.")

# Confirm the new view can be queried.
summary = con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys
    FROM a01_v3_transitions
""").df()

display(summary)

PASS: a01_v3_transitions created.


,rows,unique_keys
0,776,776


In [54]:
# ============================================================
# A01 v3 — Structural and transition coverage audit
# ============================================================

feature_names = [
    name
    for pair in transition_names.values()
    for name in pair
]

# Check row counts and uniqueness.
audit = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys
    FROM a01_v3_transitions
""").df()

assert audit.loc[0, "row_count"] == 776
assert audit.loc[0, "unique_keys"] == 776

# Check that all previous v2C columns are preserved.
old_columns = set(
    con.sql("DESCRIBE SELECT * FROM a01_v2c_rolling")
    .df()["column_name"]
)

new_columns = set(
    con.sql("DESCRIBE SELECT * FROM a01_v3_transitions")
    .df()["column_name"]
)

assert old_columns.issubset(new_columns)
assert set(feature_names).issubset(new_columns)

# Summarize observed TRUE, FALSE, and NULL values.
coverage_sql = []

for name in feature_names:
    coverage_sql.append(f"""
        SELECT
            '{name}' AS feature,
            COUNT(*) FILTER (WHERE {name} = TRUE) AS true_count,
            COUNT(*) FILTER (WHERE {name} = FALSE) AS false_count,
            COUNT(*) FILTER (WHERE {name} IS NULL) AS null_count
        FROM a01_v3_transitions
    """)

coverage = con.sql(
    " UNION ALL ".join(coverage_sql)
).df()

coverage["observed_count"] = (
    coverage["true_count"] + coverage["false_count"]
)

coverage["coverage_pct"] = (
    coverage["observed_count"] / 776 * 100
).round(1)

display(coverage)

print("PASS: 776 unique franchise-season observations.")
print("PASS: All v2C columns preserved.")
print("PASS: All 12 v3 transition features present.")

,feature,true_count,false_count,null_count,observed_count,coverage_pct
0,entered_tax_threshold,36,384,356,420,54.1
1,exited_tax_threshold,36,384,356,420,54.1
2,entered_first_apron,28,392,356,420,54.1
3,exited_first_apron,29,391,356,420,54.1
4,entered_second_apron,2,58,716,60,7.7
5,exited_second_apron,6,54,716,60,7.7
6,newly_qualified_playoffs,107,638,31,745,96.0
7,lost_playoff_qualification,107,638,31,745,96.0
8,newly_reached_second_round,91,654,31,745,96.0
9,failed_to_repeat_second_round,90,655,31,745,96.0


PASS: 776 unique franchise-season observations.
PASS: All v2C columns preserved.
PASS: All 12 v3 transition features present.


In [55]:
# ============================================================
# A01 v3 — Independent pandas arithmetic validation
# ============================================================

# Pull only the source columns and newly engineered features.
source_cols = [
    "franchise_id",
    "season_year",
    "taxable_payroll_to_tax_ratio",
    "first_apron_allocation_ratio",
    "second_apron_allocation_ratio",
    "made_playoffs",
    "reached_second_round",
    "reached_conference_finals",
]

validation = con.sql(f"""
    SELECT
        {", ".join(source_cols + feature_names)}
    FROM a01_v3_transitions
""").df()

validation = validation.sort_values(
    ["franchise_id", "season_year"]
).reset_index(drop=True)

# Independently classify financial states using pandas.
for name, ratio_col in {
    "tax_threshold": "taxable_payroll_to_tax_ratio",
    "first_apron": "first_apron_allocation_ratio",
    "second_apron": "second_apron_allocation_ratio",
}.items():

    validation[f"check_{name}"] = (
        validation[ratio_col]
        .ge(1)
        .where(validation[ratio_col].notna())
        .astype("boolean")
    )

# Competitive states already exist as Boolean source columns.
for name, source_col in competitive_states.items():
    validation[f"check_{name}"] = (
        validation[source_col].astype("boolean")
    )

# Compute previous-season values independently using groupby/shift.
previous_year = (
    validation.groupby("franchise_id")["season_year"].shift(1)
)

consecutive = previous_year.eq(
    validation["season_year"] - 1
).fillna(False)

results = []

for name, (entry_name, exit_name) in transition_names.items():

    current = validation[f"check_{name}"]

    previous = (
        validation.groupby("franchise_id")[f"check_{name}"]
        .shift(1)
        .astype("boolean")
    )

    valid = consecutive & current.notna() & previous.notna()

    expected_entry = (
        (~previous & current)
        .where(valid)
        .astype("boolean")
    )

    expected_exit = (
        (previous & ~current)
        .where(valid)
        .astype("boolean")
    )

    for feature, expected in [
        (entry_name, expected_entry),
        (exit_name, expected_exit),
    ]:

        actual = validation[feature].astype("boolean")

        # Nullable Boolean comparison:
        # fillna(False) only after comparing missingness separately.
        missing_match = actual.isna().eq(expected.isna())

        value_match = (
            actual.eq(expected)
            .fillna(True)
        )

        mismatches = (~(missing_match & value_match)).sum()

        results.append({
            "feature": feature,
            "mismatches": int(mismatches),
        })

results_df = pd.DataFrame(results)

display(results_df)

assert results_df["mismatches"].sum() == 0

print("PASS: All 12 transitions independently validated.")

,feature,mismatches
0,entered_tax_threshold,0
1,exited_tax_threshold,0
2,entered_first_apron,0
3,exited_first_apron,0
4,entered_second_apron,0
5,exited_second_apron,0
6,newly_qualified_playoffs,0
7,lost_playoff_qualification,0
8,newly_reached_second_round,0
9,failed_to_repeat_second_round,0


PASS: All 12 transitions independently validated.


In [56]:
# A01 v3 — Final semantic and historical-coverage checks

checks = con.sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE reached_conference_finals = TRUE
              AND reached_second_round IS DISTINCT FROM TRUE
        ) AS conference_finals_inconsistencies,

        COUNT(*) FILTER (
            WHERE reached_second_round = TRUE
              AND made_playoffs IS DISTINCT FROM TRUE
        ) AS second_round_inconsistencies

    FROM a01_v3_transitions
""").df()

print("Postseason consistency:")
display(checks)

coverage_by_season = con.sql("""
    SELECT
        season,
        COUNT(entered_tax_threshold) AS tax_valid,
        COUNT(entered_first_apron) AS first_apron_valid,
        COUNT(entered_second_apron) AS second_apron_valid
    FROM a01_v3_transitions
    GROUP BY season, season_year
    ORDER BY season_year
""").df()

print("Financial transition coverage by season:")
display(coverage_by_season)

assert checks.iloc[0].eq(0).all(), (
    "Postseason progression inconsistency detected."
)

print("PASS: Postseason progression is internally consistent.")

Postseason consistency:


,conference_finals_inconsistencies,second_round_inconsistencies
0,0,0


Financial transition coverage by season:


,season,tax_valid,first_apron_valid,second_apron_valid
0,2000-2001,0,0,0
1,2001-2002,0,0,0
2,2002-2003,0,0,0
3,2003-2004,0,0,0
4,2004-2005,0,0,0
5,2005-2006,0,0,0
6,2006-2007,0,0,0
7,2007-2008,0,0,0
8,2008-2009,0,0,0
9,2009-2010,0,0,0


PASS: Postseason progression is internally consistent.


In [57]:
# ============================================================
# A01 v4 — Final feature inventory and coverage audit
# ============================================================

import pandas as pd

# Retrieve the schemas of the previous and final A01 views.
v2_schema = con.sql("""
    DESCRIBE SELECT * FROM a01_v2c_rolling
""").df()

v3_schema = con.sql("""
    DESCRIBE SELECT * FROM a01_v3_transitions
""").df()

# Get row count and distinct franchise-season keys.
summary = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys,
        COUNT(DISTINCT franchise_id) AS franchises,
        COUNT(DISTINCT season) AS seasons
    FROM a01_v3_transitions
""").df()

# Identify which columns were added in v3.
previous_columns = set(v2_schema["column_name"])

new_features = v3_schema.loc[
    ~v3_schema["column_name"].isin(previous_columns),
    "column_name"
].tolist()

# Generate a SQL expression for non-null coverage
# for every column in the final analytical dataset.
coverage_expressions = [
    f'COUNT("{col}") AS "{col}"'
    for col in v3_schema["column_name"]
]

coverage_counts = con.sql(f"""
    SELECT
        {", ".join(coverage_expressions)}
    FROM a01_v3_transitions
""").df().iloc[0]

# Assemble a searchable inventory.
inventory = v3_schema[
    ["column_name", "column_type"]
].copy()

inventory["non_null_count"] = inventory["column_name"].map(
    coverage_counts.to_dict()
)

inventory["coverage_pct"] = (
    inventory["non_null_count"]
    / int(summary.loc[0, "row_count"])
    * 100
).round(1)

inventory["added_in_v3"] = (
    inventory["column_name"].isin(new_features)
)

# Final structural assertions.
assert summary.loc[0, "row_count"] == 776
assert summary.loc[0, "unique_keys"] == 776
assert len(new_features) == 12
assert previous_columns.issubset(
    set(v3_schema["column_name"])
)

print("A01 v4 — Final structural summary")
display(summary)

print(f"Total dataset columns: {len(inventory)}")
print(f"New v3 features: {len(new_features)}")

print("\nV3 feature inventory:")
display(inventory[inventory["added_in_v3"]])

print("\nCoverage distribution across all columns:")
display(
    inventory["coverage_pct"]
    .describe()
    .round(1)
    .to_frame("coverage_pct")
)

print("\nPASS: A01 final inventory and structure validated.")

A01 v4 — Final structural summary


,row_count,unique_keys,franchises,seasons
0,776,776,30,26


Total dataset columns: 210
New v3 features: 12

V3 feature inventory:


,column_name,column_type,non_null_count,coverage_pct,added_in_v3
198,entered_tax_threshold,BOOLEAN,420,54.1,True
199,exited_tax_threshold,BOOLEAN,420,54.1,True
200,entered_first_apron,BOOLEAN,420,54.1,True
201,exited_first_apron,BOOLEAN,420,54.1,True
202,entered_second_apron,BOOLEAN,60,7.7,True
203,exited_second_apron,BOOLEAN,60,7.7,True
204,newly_qualified_playoffs,BOOLEAN,745,96.0,True
205,lost_playoff_qualification,BOOLEAN,745,96.0,True
206,newly_reached_second_round,BOOLEAN,745,96.0,True
207,failed_to_repeat_second_round,BOOLEAN,745,96.0,True



Coverage distribution across all columns:


,coverage_pct
count,210.0
mean,69.2
std,30.7
min,3.9
25%,54.1
50%,58.0
75%,100.0
max,100.0



PASS: A01 final inventory and structure validated.


In [58]:
# ============================================================
# A01 v4 — Fix Parquet count types and verify export
# ============================================================

from pathlib import Path

# Inspect the final source schema directly; do not depend on a failed exploratory export.
source_schema = con.sql("DESCRIBE SELECT * FROM a01_v3_transitions").df()

# DuckDB aggregates can be HUGEINT; Parquet should store these counts as BIGINT.
count_columns = source_schema.loc[
    source_schema["column_type"] == "HUGEINT", "column_name"
].tolist()

assert len(count_columns) == 11, f"Expected 11 HUGEINT counts; found {count_columns}"

# Each column counts events across a 3-year or 6-year window.
# Counts must be whole numbers between zero and window length.
for col in count_columns:

    max_count = 3 if col.endswith("_3yr") else 6

    check = con.sql(f"""
        SELECT COUNT(*) AS invalid_count
        FROM a01_v3_transitions
        WHERE "{col}" IS NOT NULL
          AND (
              "{col}" < 0
              OR "{col}" > {max_count}
          )
    """).df()

    assert check.loc[0, "invalid_count"] == 0, (
        f"Invalid count values in {col}"
    )

print("PASS: All 11 rolling-count columns are within expected bounds.")

# Locate the project root and final output directory.
cwd = Path.cwd()

if (cwd / "data" / "canonical").is_dir():
    project_root = cwd
elif (cwd.parent / "data" / "canonical").is_dir():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Cannot locate project root.")

output_path = (
    project_root / "data" / "analytical"
    / "a01_franchise_season.parquet"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

# Preserve original column order while converting the
# 11 rolling-count columns to BIGINT.
select_expressions = []

for col in source_schema["column_name"]:

    if col in count_columns:
        select_expressions.append(
            f'CAST("{col}" AS BIGINT) AS "{col}"'
        )
    else:
        select_expressions.append(f'"{col}"')

# Export a new version of the Parquet file.
# This replaces the earlier export that had type mismatches.
con.execute(f"""
    COPY (
        SELECT {", ".join(select_expressions)}
        FROM a01_v3_transitions
        ORDER BY franchise_id, season_year
    )
    TO '{str(output_path).replace(chr(39), chr(39) * 2)}'
    (FORMAT PARQUET, COMPRESSION ZSTD, OVERWRITE_OR_IGNORE TRUE)
""").fetchall()

# Reload and inspect the persisted file.
saved_schema_fixed = con.cursor().execute("""
    DESCRIBE SELECT *
    FROM read_parquet(?)
""", [str(output_path)]).df()

# Build the expected schema: BIGINT for the 11 count columns,
# unchanged types for all remaining columns.
expected_types = source_schema["column_type"].tolist()

expected_types = [
    "BIGINT" if col in count_columns else dtype
    for col, dtype in zip(
        source_schema["column_name"],
        expected_types
    )
]

assert (
    saved_schema_fixed["column_type"].tolist()
    == expected_types
), "Parquet schema still differs from expected types."

# Check row count, column count and unique franchise-season keys.
saved_check = con.cursor().execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(*) - COUNT(DISTINCT (franchise_id, season))
            AS duplicate_keys
    FROM read_parquet(?)
""", [str(output_path)]).df()

assert saved_check.loc[0, "rows"] == 776
assert saved_check.loc[0, "duplicate_keys"] == 0
assert len(saved_schema_fixed) == 210

# Independently reconcile the persisted file against the source view.
# Compare every non-floating column exactly; floating-point columns use
# a small relative/absolute tolerance to accommodate serialization rounding.
# We join by the validated unique franchise-season key, so row order is irrelevant.

float_types = {"FLOAT", "DOUBLE", "REAL"}
column_names = source_schema["column_name"].tolist()
column_types = source_schema["column_type"].tolist()

# The key columns must be present on both sides of the comparison.
key_columns = ["franchise_id", "season"]
assert all(key in column_names for key in key_columns)

# Cast the 11 count columns in the source to the same exported types.
# The saved Parquet schema was already checked above.
source_projection = ", ".join(select_expressions)

# Quote identifiers to handle column names safely.
def quote_ident(name):
    return '"' + name.replace('"', '""') + '"'

# Test each value, including NULLs. IS NOT DISTINCT FROM treats two NULLs as equal.
# For FLOAT/DOUBLE, a difference <= 1e-10 * max(1, |a|, |b|) is tolerated.
comparisons = []
for name, dtype in zip(column_names, column_types):
    if name in key_columns:
        continue
    quoted = quote_ident(name)
    left, right = f's.{quoted}', f'p.{quoted}'
    if dtype.upper() in float_types:
        same = (
            f'({left} IS NOT DISTINCT FROM {right} OR '
            f'({left} IS NOT NULL AND {right} IS NOT NULL AND '
            f'ABS({left} - {right}) <= '
            f'1e-10 * GREATEST(1.0, ABS({left}), ABS({right}))))'
        )
    else:
        same = f'({left} IS NOT DISTINCT FROM {right})'
    comparisons.append((name, same))

join_condition = " AND ".join(
    f's.{quote_ident(key)} IS NOT DISTINCT FROM p.{quote_ident(key)}'
    for key in key_columns
)

# A FULL OUTER JOIN also detects keys present on only one side.
# A non-null marker distinguishes unmatched rows from matched rows with NULL values.
per_column_sql = ",\n".join(
    f'SUM(CASE WHEN NOT {condition} THEN 1 ELSE 0 END) AS {quote_ident(name)}'
    for name, condition in comparisons
)

comparison_sql = f"""
WITH s AS (
    SELECT 1 AS _source_present, {source_projection}
    FROM a01_v3_transitions
),
p AS (
    SELECT 1 AS _parquet_present, *
    FROM read_parquet('{str(output_path).replace(chr(39), chr(39) * 2)}')
)
SELECT
    COUNT(*) FILTER (WHERE s._source_present IS NULL OR p._parquet_present IS NULL)
        AS unmatched_keys,
    {per_column_sql}
FROM s FULL OUTER JOIN p ON {join_condition}
"""

comparison_results = con.sql(comparison_sql).df().iloc[0]
unmatched_keys = int(comparison_results["unmatched_keys"])
material_mismatches = {
    name: int(comparison_results[name])
    for name, _ in comparisons
    if int(comparison_results[name]) > 0
}

print("Unmatched franchise-season keys:", unmatched_keys)
print("Columns with material value mismatches:", material_mismatches)
assert unmatched_keys == 0, "Source and Parquet contain different franchise-season keys."
assert not material_mismatches, f"Material Parquet differences: {material_mismatches}"

print("PASS: Parquet export validated using exact and float-tolerant comparisons.")
print("PASS: All 210 column types match the expected export schema.")
print("PASS: All 776 rows preserved without material value mismatches.")
print("PASS: All 11 rolling-count columns stored as BIGINT.")
print("File:", output_path)
print("Size (MB):", round(output_path.stat().st_size / 1024**2, 2))


PASS: All 11 rolling-count columns are within expected bounds.
Unmatched franchise-season keys: 0
Columns with material value mismatches: {}
PASS: Parquet export validated using exact and float-tolerant comparisons.
PASS: All 210 column types match the expected export schema.
PASS: All 776 rows preserved without material value mismatches.
PASS: All 11 rolling-count columns stored as BIGINT.
File: /Users/jonathantran/Desktop/nba_economics/data/analytical/a01_franchise_season.parquet
Size (MB): 0.33


In [59]:
# ============================================================
# A01 v4 — Diagnose persisted data differences
# ============================================================

# Use the source view and saved Parquet file.
# The previous export has already been written successfully.

source_df = con.sql("""
    SELECT *
    FROM a01_v3_transitions
    ORDER BY franchise_id, season_year
""").df()

saved_df = con.cursor().execute("""
    SELECT *
    FROM read_parquet(?)
    ORDER BY franchise_id, season_year
""", [str(output_path)]).df()

# Verify that observations are aligned before comparing values.
keys = ["franchise_id", "season"]

assert source_df[keys].equals(saved_df[keys])
assert source_df.columns.tolist() == saved_df.columns.tolist()

results = []

for col in source_df.columns:

    original = source_df[col]
    saved = saved_df[col]

    # Identify missing-value differences separately.
    missing_mismatch = (
        original.isna() != saved.isna()
    )

    # Compare values only where both are nonmissing.
    both_present = original.notna() & saved.notna()

    # Use exact equality first to identify discrepancies.
    exact_mismatch = pd.Series(
        False,
        index=original.index
    )

    exact_mismatch.loc[both_present] = (
        original.loc[both_present]
        != saved.loc[both_present]
    )

    # For numeric columns, measure the largest absolute difference.
    max_abs_diff = None

    if (
        pd.api.types.is_numeric_dtype(original)
        and pd.api.types.is_numeric_dtype(saved)
    ):
        numeric_diff = (
            original.loc[both_present].astype("float64")
            - saved.loc[both_present].astype("float64")
        ).abs()

        max_abs_diff = (
            numeric_diff.max()
            if not numeric_diff.empty
            else None
        )

    results.append({
        "column": col,
        "value_mismatches": int(exact_mismatch.sum()),
        "null_mismatches": int(missing_mismatch.sum()),
        "max_abs_diff": max_abs_diff,
    })

diagnostics = pd.DataFrame(results)

problem_columns = diagnostics[
    (diagnostics["value_mismatches"] > 0)
    | (diagnostics["null_mismatches"] > 0)
]

print("Columns with detected differences:", len(problem_columns))
display(problem_columns)

print(
    "\nTotal value mismatches:",
    diagnostics["value_mismatches"].sum()
)

print(
    "Total NULL mismatches:",
    diagnostics["null_mismatches"].sum()
)

Columns with detected differences: 3


,column,value_mismatches,null_mismatches,max_abs_diff
127,win_pct_vs_payroll_quartile_mean,97,0,1.665335e-16
131,win_pct_vs_taxable_payroll_quartile_mean,142,0,1.110223e-16
135,win_pct_vs_apron_allocations_quartile_mean,151,0,1.110223e-16



Total value mismatches: 390
Total NULL mismatches: 0


In [60]:
# ============================================================
# A01 v4 — Final type-aware Parquet validation
# ============================================================

import numpy as np
import pandas as pd

# Confirm structure and schema before checking individual values.
assert source_df.shape == saved_df.shape == (776, 210)

assert source_df.columns.tolist() == saved_df.columns.tolist()

assert (
    saved_schema_fixed["column_name"].tolist()
    == source_schema["column_name"].tolist()
)

# These 11 rolling counts were intentionally normalized
# from HUGEINT to BIGINT during export.
expected_types = [
    "BIGINT" if col in count_columns else dtype
    for col, dtype in zip(
        source_schema["column_name"],
        source_schema["column_type"]
    )
]

assert saved_schema_fixed["column_type"].tolist() == expected_types

# The dataframes were previously sorted by franchise and season.
assert source_df[["franchise_id", "season"]].equals(
    saved_df[["franchise_id", "season"]]
)

failures = []

for col, dtype in zip(
    source_schema["column_name"],
    source_schema["column_type"]
):

    original = source_df[col]
    saved = saved_df[col]

    # Missingness must match exactly.
    if not original.isna().equals(saved.isna()):
        failures.append((col, "NULL mismatch"))
        continue

    # Compare only values that are present in both datasets.
    valid = original.notna()

    if dtype in ("FLOAT", "DOUBLE"):

        # Floating-point calculations may differ slightly
        # after writing and reading Parquet.
        matches = np.allclose(
            original.loc[valid].to_numpy(dtype="float64"),
            saved.loc[valid].to_numpy(dtype="float64"),
            rtol=0,
            atol=1e-12,
            equal_nan=False
        )

    else:

        # Integers, Booleans, strings, and dates must match exactly.
        matches = original.loc[valid].equals(
            saved.loc[valid]
        )

        # For intentionally normalized HUGEINT -> BIGINT columns,
        # compare exact integer values without requiring identical
        # pandas dtype representations.
        if col in count_columns:
            matches = (
                original.loc[valid].tolist()
                == saved.loc[valid].tolist()
            )

    if not matches:
        failures.append((col, "Value mismatch"))

assert not failures, f"Validation failures: {failures}"

print("PASS: Final A01 Parquet validation complete.")
print("PASS: 776 rows and 210 columns preserved.")
print("PASS: All 11 rolling-count columns normalized to BIGINT.")
print("PASS: All NULL patterns preserved.")
print("PASS: Exact values preserved outside floating-point columns.")
print("PASS: Floating-point values agree within 1e-12.")
print("File:", output_path)
print("Size (MB):", round(output_path.stat().st_size / 1024**2, 2))

PASS: Final A01 Parquet validation complete.
PASS: 776 rows and 210 columns preserved.
PASS: All 11 rolling-count columns normalized to BIGINT.
PASS: All NULL patterns preserved.
PASS: Exact values preserved outside floating-point columns.
PASS: Floating-point values agree within 1e-12.
File: /Users/jonathantran/Desktop/nba_economics/data/analytical/a01_franchise_season.parquet
Size (MB): 0.33


In [61]:
# ============================================================
# A01 v4 — Generate final column inventory
# ============================================================

from pathlib import Path
import pandas as pd

# Locate the project root.
cwd = Path.cwd()

if (cwd / "data" / "canonical").is_dir():
    project_root = cwd
elif (cwd.parent / "data" / "canonical").is_dir():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Cannot locate project root.")

# Read the finalized Parquet file.
parquet_path = (
    project_root / "data" / "analytical"
    / "a01_franchise_season.parquet"
)

# DuckDB reads Parquet directly without requiring pyarrow.
# .df() converts the DuckDB query result into a pandas DataFrame.
df = con.cursor().execute(
    "SELECT * FROM read_parquet(?)",
    [str(parquet_path)]
).df()

# Read persisted DuckDB types directly from the file.
schema = con.cursor().execute("""
    DESCRIBE SELECT *
    FROM read_parquet(?)
""", [str(parquet_path)]).df()

# Calculate how often each column contains a usable value.
inventory = pd.DataFrame({
    "column_name": df.columns,
    "non_null_count": df.notna().sum().to_numpy(),
    "coverage_pct": (
        df.notna().mean().to_numpy() * 100
    ).round(1),
})

# Attach the authoritative persisted data types.
inventory = inventory.merge(
    schema[["column_name", "column_type"]],
    on="column_name",
    how="left",
    validate="one_to_one"
)

# Restore the intended column ordering.
inventory = inventory[
    ["column_name", "column_type",
     "non_null_count", "coverage_pct"]
]

# Mark normalized rolling-count columns.
inventory["storage_note"] = inventory["column_name"].apply(
    lambda name: "HUGEINT normalized to BIGINT for Parquet"
    if name in count_columns else ""
)

# Leave semantic fields blank until we reconcile
# the previous feature library with the final SQL.
for field in [
    "feature_family",
    "definition",
    "calculation_logic",
    "source_dependencies",
    "limitations"
]:
    inventory[field] = ""

assert len(inventory) == 210
assert inventory["column_name"].is_unique
assert inventory["column_type"].notna().all()

# Save the working inventory in docs/.
docs_dir = project_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

inventory_path = docs_dir / "a01_feature_library.csv"

inventory.to_csv(inventory_path, index=False)

print("PASS: Final A01 column inventory generated.")
print("Columns documented:", len(inventory))
print("File:", inventory_path)

display(inventory.head(10))

PASS: Final A01 column inventory generated.
Columns documented: 210
File: /Users/jonathantran/Desktop/nba_economics/docs/a01_feature_library.csv


,column_name,column_type,non_null_count,coverage_pct,storage_note,feature_family,definition,calculation_logic,source_dependencies,limitations
0,team_name,VARCHAR,776,100.0,,,,,,
1,wins,BIGINT,776,100.0,,,,,,
2,losses,BIGINT,776,100.0,,,,,,
3,win_pct,DOUBLE,776,100.0,,,,,,
4,games_behind,DOUBLE,776,100.0,,,,,,
5,points_per_game,DOUBLE,776,100.0,,,,,,
6,opponent_points_per_game,DOUBLE,776,100.0,,,,,,
7,simple_rating_system,DOUBLE,776,100.0,,,,,,
8,conference,VARCHAR,776,100.0,,,,,,
9,season,VARCHAR,776,100.0,,,,,,
